# OfficePulse Growth Analytics Project

- **Author:** Emmanuel Nti
- **Date:** August 9, 2026

## Project Overview

This Growth Analytics project establishes a reliable analytical foundation for understanding the end-to-end customer journey for a fictional SaaS platform that helps organisations manage workplaces and optimise office space usage.

Using three fictional business datasets — **Paid Ads, CRM Pipeline, and Product Usage Events** — the project combines **analytics engineering and exploratory analysis** to develop scalable data models, define meaningful growth metrics, and identify performance drivers and bottlenecks across **marketing acquisition, funnel conversion, revenue performance, lead cohort quality, and product engagement**.

## Analytical Approach

The analysis follows 5 key steps:

1. **Define the analytical framework**

   - Review the business context and available datasets.
   - Identify the most relevant growth metrics and document the rationale for their selection.

2. **Build the data models**

   - Transform the raw datasets into analysis-ready data marts using dbt.
   - Validate data model quality through automated testing to ensure consistency and reliability.
   - **Interactive dbt documentation:** [https://growth-analytics-dbt-docs.netlify.app/#!/overview](https://growth-analytics-dbt-docs.netlify.app/#!/overview)

3. **Perform exploratory data analysis**

   - Analyze overall business performance, performance trends, acquisition performance, funnel conversion, lead cohort quality, and customer product engagement.
   - Identify key trends, patterns, bottlenecks, and opportunities across the customer journey.

4. **Summarize findings and recommendations**

   - Consolidate the key findings from the analysis.
   - Present recommendations supported by the analytical evidence.

5. **Document limitations and future enhancements**

   - Document the data and analytical limitations that influence the interpretation of the results.
   - Identify future enhancements that could be supported with additional business data.

## Notebook Structure

The remainder of this notebook follows the analytical approach described above and is organized into the following sections:

- **Step 1: Data Validation and Inspection**
- **Step 2: Executive KPI Overview**
- **Step 3: Performance Trends**
- **Step 4: Acquisition Performance**
- **Step 5: Acquisition Conversion Performance**
- **Step 6: Lead Cohort Quality**
- **Step 7: Customer Product Engagement**
- **Step 8: Summary of Findings and Recommendations**
- **Step 9: Limitations and Future Enhancements**


## Step 1: Data Validation and Inspection
- Review dataset dimensions, sample records, and coverage of each source dataset to verify the raw data and analytical marts before beginning the exploratory analysis.

In [1]:
# =====================================
# Import Libraries
# =====================================

import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy
import plotly.io as pio
pio.renderers.default = "notebook_connected" 
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from datetime import datetime

pipeline_start = datetime.now()
pd.set_option("display.max_columns", None)

# -------------------------------------
# Connect to DuckDB
# -------------------------------------

db_path = Path("../data/officepulse.duckdb").resolve()

conn = duckdb.connect(
    str(db_path),
    read_only=True
)

#print(f"Connected to: {db_path}")
print("Inspecting the loaded raw tables and analytical marts before the exploratory analysis.\n")
# -------------------------------------
# Load raw tables
# -------------------------------------

raw_paid_ads = conn.sql(
    "SELECT * FROM raw_paid_ads"
).df()

raw_crm_pipeline = conn.sql(
    "SELECT * FROM raw_crm_pipeline"
).df()

raw_product_usage = conn.sql(
    "SELECT * FROM raw_product_usage_events"
).df()

# -------------------------------------
# Load analytical marts
# -------------------------------------

campaign_performance = conn.sql(
    "SELECT * FROM mart_campaign_performance"
).df()

lead_cohorts = conn.sql(
    "SELECT * FROM mart_lead_cohort_progression"
).df()

product_adoption = conn.sql(
    "SELECT * FROM mart_product_adoption"
).df()



# -------------------------------------
# Dataset summary
# -------------------------------------

datasets = {
    "Raw Paid Ads": raw_paid_ads,
    "Raw CRM Pipeline": raw_crm_pipeline,
    "Raw Product Usage": raw_product_usage,
    "Campaign Performance Mart": campaign_performance,
    "Lead Cohort Mart": lead_cohorts,
    "Product Adoption Mart": product_adoption,
}

summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()],
})

display(summary)

# -------------------------------------
# Preview sample records
# -------------------------------------

for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head(2))


# ============================================================
# 1.2 DATA COVERAGE VALIDATION
# Compare date ranges across source datasets to understand
# the data coverage and ensure metrics are interpreted correctly.
# ============================================================
print("Comparing date ranges across source datasets to understand the data coverage and ensure metrics are interpreted correctly.\n")
date_range_checks = []

# Paid ads
paid_ads_range = conn.execute("""
    SELECT
        'Paid Ads' AS table_name,
        'date' AS date_field,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(*) AS row_count
    FROM raw_paid_ads
""").df()

date_range_checks.append(paid_ads_range)


# CRM: lead creation
crm_created_range = conn.execute("""
    SELECT
        'CRM Pipeline' AS table_name,
        'created_at' AS date_field,
        MIN(created_at) AS min_date,
        MAX(created_at) AS max_date,
        COUNT(created_at) AS row_count
    FROM raw_crm_pipeline
""").df()

date_range_checks.append(crm_created_range)


# CRM: opportunity creation
crm_opportunity_range = conn.execute("""
    SELECT
        'CRM Pipeline' AS table_name,
        'opportunity_created_at' AS date_field,
        MIN(opportunity_created_at) AS min_date,
        MAX(opportunity_created_at) AS max_date,
        COUNT(opportunity_created_at) AS row_count
    FROM raw_crm_pipeline
""").df()

date_range_checks.append(crm_opportunity_range)


# CRM: close date
crm_close_range = conn.execute("""
    SELECT
        'CRM Pipeline' AS table_name,
        'close_date' AS date_field,
        MIN(close_date) AS min_date,
        MAX(close_date) AS max_date,
        COUNT(close_date) AS row_count
    FROM raw_crm_pipeline
""").df()

date_range_checks.append(crm_close_range)


# CRM: trial start
crm_trial_start_range = conn.execute("""
    SELECT
        'CRM Pipeline' AS table_name,
        'trial_start_at' AS date_field,
        MIN(trial_start_at) AS min_date,
        MAX(trial_start_at) AS max_date,
        COUNT(trial_start_at) AS row_count
    FROM raw_crm_pipeline
""").df()

date_range_checks.append(crm_trial_start_range)


# CRM: trial end
crm_trial_end_range = conn.execute("""
    SELECT
        'CRM Pipeline' AS table_name,
        'trial_end_at' AS date_field,
        MIN(trial_end_at) AS min_date,
        MAX(trial_end_at) AS max_date,
        COUNT(trial_end_at) AS row_count
    FROM raw_crm_pipeline
""").df()

date_range_checks.append(crm_trial_end_range)


# Product usage events
product_events_range = conn.execute("""
    SELECT
        'Product Usage Events' AS table_name,
        'event_time' AS date_field,
        MIN(event_time) AS min_date,
        MAX(event_time) AS max_date,
        COUNT(event_time) AS row_count
    FROM raw_product_usage_events
""").df()

date_range_checks.append(product_events_range)


date_range_summary = pd.concat(date_range_checks, ignore_index=True)

display(date_range_summary)


# =====================================
# 1.3 Dataset Overview
# =====================================

print("Summarising the scope of the datasets used in the analysis.\n")

dataset_overview = pd.DataFrame({
    "Metric": [
        "Date Range",
        "Total Campaigns",
        "Total Ad Spend",
        "Total Leads",
        "Total Opportunities",
        "Total Customers",
        "Unique Companies",
        "Unique Users",
        "Total Product Events"
    ],
    "Value": [
        f"{min(raw_paid_ads['date'].min(), raw_crm_pipeline['created_at'].min(), raw_product_usage['event_time'].min()).date()} "
        f"to "
        f"{max(raw_paid_ads['date'].max(), raw_crm_pipeline['created_at'].max(), raw_product_usage['event_time'].max()).date()}",
        raw_paid_ads["campaign_id"].nunique(),
        f"€{raw_paid_ads['cost'].sum():,.0f}",
        raw_crm_pipeline["lead_id"].nunique(),
        raw_crm_pipeline["opportunity_id"].dropna().nunique(),
        raw_crm_pipeline.loc[
    raw_crm_pipeline["close_status"] == "Won",
    "company_id"
].nunique(),
        pd.concat([
            raw_crm_pipeline["company_id"],
            raw_product_usage["company_id"]
        ]).nunique(),
        raw_product_usage["user_id"].nunique(),
        len(raw_product_usage)
    ]
})

display(dataset_overview)

print(
    "Note: The dataset does not specify a currency. EUR (€) is assumed for presentation purposes given officepulse is a European company.\n"
)

Inspecting the loaded raw tables and analytical marts before the exploratory analysis.



,Dataset,Rows,Columns
0,Raw Paid Ads,5187,10
1,Raw CRM Pipeline,500,15
2,Raw Product Usage,33498,6
3,Campaign Performance Mart,5187,25
4,Lead Cohort Mart,9,13
5,Product Adoption Mart,62,5



Raw Paid Ads


,date,campaign_id,campaign_name,channel,utm_source,utm_medium,utm_campaign,impressions,clicks,cost
0,2025-01-01,CMP_1001,EMEA_FeatureY_Search,Paid Social,linkedin,paid_social,camp_1,2948,61,482.13
1,2025-01-01,CMP_1002,APAC_Brand_Search,Paid Social,linkedin,paid_social,camp_2,4198,74,384.27



Raw CRM Pipeline


...

Comparing date ranges across source datasets to understand the data coverage and ensure metrics are interpreted correctly.



,table_name,date_field,min_date,max_date,row_count
0,Paid Ads,date,2025-01-01 00:00:00,2025-09-30 00:00:00,5187
1,CRM Pipeline,created_at,2025-01-01 00:00:00,2025-09-30 00:00:00,500
2,CRM Pipeline,opportunity_created_at,2025-01-15 00:00:00,2025-11-07 00:00:00,162
3,CRM Pipeline,close_date,2025-02-05 00:00:00,2026-01-17 00:00:00,162
4,CRM Pipeline,trial_start_at,2025-01-02 00:00:00,2025-10-08 00:00:00,311
5,CRM Pipeline,trial_end_at,2025-01-16 00:00:00,2025-10-22 00:00:00,311
6,Product Usage Events,event_time,2025-01-02 06:35:00,2026-01-14 22:54:00,33498


Summarising the scope of the datasets used in the analysis.



,Metric,Value
0,Date Range,2025-01-01 to 2026-01-14
1,Total Campaigns,19
2,Total Ad Spend,"€1,557,873"
3,Total Leads,500
4,Total Opportunities,162
5,Total Customers,71
6,Unique Companies,500
7,Unique Users,5635
8,Total Product Events,33498


Note: The dataset does not specify a currency. EUR (€) is assumed for presentation purposes given officepulse is a European company.



##### Note on Trial Activation Metrics and Data Coverage

- The lead cohort mart includes **trials started**, **activated trials**, and **trial activation rate**. During exploratory analysis, trial activation was found to be **100% across all months**, indicating no variation in the dataset.
    - As a result, these metrics do not provide meaningful analytical insight or support comparisons over time.
    - The subsequent analysis therefore focuses on funnel conversion and commercial KPIs that exhibit meaningful variation.
- Marketing campaign data is available through September 2025, so performance trends are shown through that period. CRM close dates and product usage events extend beyond September 2025 and are analyzed separately where relevant. 


## Step 2: Executive KPI Overview

In [2]:
# ============================================================
# CUSTOMISED PLOTLY TEMPLATE
# Define once and reuse in all subsequent visualisations
# ============================================================

CHART_STYLE = {
    "font": {
        "family": "Calibri",
        "size": 14,
        "color": "#2F2F2F",
    },
    "title": {
        "font": {
            "family": "Calibri",
            "size": 22,
            "color": "#2F2F2F",
        },
        "x": 0.5,
        "xanchor": "center",
        "y": 0.94,
        "yanchor": "top",
    },
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "margin": {
        "l": 30,
        "r": 30,
        "t": 60,
        "b": 20,
    },
    "hoverlabel": {
        "font": {
            "family": "Calibri",
            "size": 13,
        }
    },
    "legend": {
        "font": {
            "family": "Calibri",
            "size": 13,
        },
        "title": {
            "font": {
                "family": "Calibri",
                "size": 14,
            }
        },
    },
}


# ------------------------------------------------------------
# 2.1 Query executive KPIs
# ------------------------------------------------------------

executive_kpis = conn.execute(
    """
    SELECT
        ROUND(SUM(total_ad_spend), 2) AS total_ad_spend,
        SUM(total_leads) AS total_leads,
        ROUND(SUM(total_pipeline_value), 2) AS pipeline_value,
        ROUND(SUM(total_won_revenue), 2) AS won_revenue,

        ROUND(
            SUM(total_won_revenue)
            / NULLIF(SUM(total_ad_spend), 0),
            2
        ) AS paid_roas,

        ROUND(
            SUM(total_ad_spend)
            / NULLIF(SUM(won_opportunities), 0),
            2
        ) AS cost_per_won_opportunity,

        ROUND(
            SUM(total_opportunities)
            / NULLIF(SUM(total_leads), 0),
            4
        ) AS lead_to_opportunity_rate,

        ROUND(
            SUM(won_opportunities)
            / NULLIF(SUM(total_opportunities), 0),
            4
        ) AS opportunity_win_rate

    FROM mart_campaign_performance
    """
).df()

kpis = executive_kpis.iloc[0]


# ------------------------------------------------------------
# 2.2 Create KPI dashboard
# ------------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=4,
    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
        ],
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"},
        ],
    ],
    horizontal_spacing=0.03,
    vertical_spacing=0.08,
)


# Top row — scale and business outcomes

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["total_ad_spend"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Total Ad Spend"
                "</span>"
            )
        },
        number={
            "prefix": "€",
            "valueformat": ",.0f",
            "font": {"size": 28},
        },
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["total_leads"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Total Leads"
                "</span>"
            )
        },
        number={
            "valueformat": ",.0f",
            "font": {"size": 28},
        },
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["pipeline_value"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Pipeline Value"
                "</span>"
            )
        },
        number={
            "prefix": "€",
            "valueformat": ",.0f",
            "font": {"size": 28},
        },
    ),
    row=1,
    col=3,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["won_revenue"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Won Revenue"
                "</span>"
            )
        },
        number={
            "prefix": "€",
            "valueformat": ",.0f",
            "font": {"size": 28},
        },
    ),
    row=1,
    col=4,
)


# Bottom row — efficiency and conversion

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["paid_roas"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Paid ROAS"
                "</span>"
            )
        },
        number={
            "suffix": "x",
            "valueformat": ".2f",
            "font": {"size": 28},
        },
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["cost_per_won_opportunity"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Cost per Won Opportunity"
                "</span>"
            )
        },
        number={
            "prefix": "€",
            "valueformat": ",.0f",
            "font": {"size": 28},
        },
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["lead_to_opportunity_rate"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Lead → Opportunity Rate"
                "</span>"
            )
        },
        number={
            "valueformat": ".1%",
            "font": {"size": 28},
        },
    ),
    row=2,
    col=3,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=kpis["opportunity_win_rate"],
        title={
            "text": (
                "<span style='font-size:15px'>"
                "Opportunity Win Rate"
                "</span>"
            )
        },
        number={
            "valueformat": ".1%",
            "font": {"size": 28},
        },
    ),
    row=2,
    col=4,
)


# ------------------------------------------------------------
# 2.3 Apply shared template
# ------------------------------------------------------------

dashboard_layout = deepcopy(CHART_STYLE)

dashboard_layout["title"]["text"] = (
    "<b>Executive KPI Overview</b>"
)

fig.update_layout(
    **dashboard_layout,
    height=300,
)

fig.show()

##### Key Findings

- The business generated **€1.39M in Won Revenue** from **€1.56M in advertising spend**, resulting in a **Paid ROAS of 0.89x**. While marketing generated a substantial **€3.06M pipeline**, attributed returns remained below marketing investment, indicating there is opportunity to improve overall marketing efficiency.
    - *This should be interpreted cautiously, as recent leads have had less time to progress through the sales pipeline.*
- The acquisition funnel converted **467 leads** into opportunities at a **33.2% Lead → Opportunity Rate**, while opportunities converted into customers at a stronger **43.9% Opportunity → Customer Conversion Rate** (Opportinutiy Win Rate). This indicates that the largest opportunity lies earlier in the funnel, where improving the progression of leads into qualified opportunities could increase the volume of sales-ready opportunities.
- Overall, the business is consistently generating a substantial pipeline while converting qualified opportunities effectively. The primary opportunity is therefore to increase the proportion of leads progressing into qualified opportunities while increasing the proportion of pipeline that ultimately converts into Won Revenue, thereby maximizing the return on marketing investment.

## Step 3: Performance Trends

In [3]:
# ------------------------------------------------------------
# 3.1 Query monthly performance trends
# ------------------------------------------------------------
performance_trends = conn.execute(
    """
    SELECT
        DATE_TRUNC('month', campaign_date) AS month,

        ROUND(
            SUM(total_ad_spend),
            2
        ) AS total_ad_spend,

        ROUND(
            SUM(total_won_revenue),
            2
        ) AS won_revenue,

        ROUND(
            SUM(total_pipeline_value),
            2
        ) AS pipeline_value,

        ROUND(
            SUM(total_won_revenue)
            / NULLIF(SUM(total_ad_spend), 0),
            2
        ) AS paid_roas,

        ROUND(
            SUM(total_ad_spend)
            / NULLIF(SUM(won_opportunities), 0),
            2
        ) AS cost_per_won_opportunity,

        ROUND(
            SUM(total_opportunities)
            / NULLIF(SUM(total_leads), 0),
            4
        ) AS lead_to_opportunity_rate,

        ROUND(
            SUM(won_opportunities)
            / NULLIF(SUM(total_opportunities), 0),
            4
        ) AS opportunity_win_rate

    FROM mart_campaign_performance

    GROUP BY 1

    ORDER BY 1
    """
).df()

performance_trends["month"] = (
    performance_trends["month"]
    .astype("datetime64[ns]")
)


# ------------------------------------------------------------
# 3.2 Define chart colours and shared formatting
# ------------------------------------------------------------

chart_colors = [
    "#636EFA",
    "#EF553B",
    "#00CC96",
    "#AB63FA",
    "#FFA15A",
    "#19D3F3",
    "#FF6692",
]

total_ad_spend_color = chart_colors[0]
cost_per_won_opportunity_color = chart_colors[1]
won_revenue_color = chart_colors[2]
pipeline_value_color = chart_colors[3]
paid_roas_color = chart_colors[4]
opportunity_win_rate_color = chart_colors[5]
lead_to_opportunity_rate_color = chart_colors[6]


def style_xaxis(fig):
    fig.update_xaxes(
        title_text=None,
        tickformat="%b %Y",
        tickangle=-45,
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=6,
        tickwidth=1,
        tickcolor="#BDBDBD",
    )


def style_layout(fig, title, height=450):
    layout = deepcopy(CHART_STYLE)

    layout["title"]["text"] = title

    layout["margin"] = {
        "l": 60,
        "r": 60,
        "t": 100,
        "b": 60,
    }

    layout["legend"] = {
        **layout.get("legend", {}),
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": 1.10,
        "yanchor": "top",
    }

    fig.update_layout(
        **layout,
        height=height,
        hovermode="x unified",
    )

    fig.update_yaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )


# ------------------------------------------------------------
# 3.3 Marketing Investment
# ------------------------------------------------------------

fig_ad_spend = go.Figure()

fig_ad_spend.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["total_ad_spend"],
        name="Total Ad Spend",
        mode="lines+markers",
        line={
            "width": 3,
            "color": total_ad_spend_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": total_ad_spend_color,
        },
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Ad Spend: €%{y:,.0f}"
            "<extra></extra>"
        ),
    )
)

style_xaxis(fig_ad_spend)

fig_ad_spend.update_yaxes(
    title_text="Ad Spend (€)",
    tickprefix="€",
    tickformat="~s",
)

style_layout(
    fig_ad_spend,
    "Marketing Investment",
)

fig_ad_spend.show()


# ------------------------------------------------------------
# 3.4 Pipeline Value
# ------------------------------------------------------------

fig_pipeline = go.Figure()

fig_pipeline.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["pipeline_value"],
        name="Pipeline Value",
        mode="lines+markers",
        line={
            "width": 3,
            "color": pipeline_value_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": pipeline_value_color,
        },
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Pipeline Value: €%{y:,.0f}"
            "<extra></extra>"
        ),
    )
)

style_xaxis(fig_pipeline)

fig_pipeline.update_yaxes(
    title_text="Pipeline Value (€)",
    tickprefix="€",
    tickformat="~s",
)

style_layout(
    fig_pipeline,
    "Pipeline Value",
)

fig_pipeline.show()


# ------------------------------------------------------------
# 3.5 Business Performance
# Won Revenue + Paid ROAS + Cost per Won Opportunity
# ------------------------------------------------------------

fig_business = go.Figure()

# Won Revenue — left axis
fig_business.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["won_revenue"],
        name="Won Revenue",
        mode="lines+markers",
        line={
            "width": 3,
            "color": won_revenue_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": won_revenue_color,
        },
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Won Revenue: €%{y:,.0f}"
            "<extra></extra>"
        ),
        yaxis="y",
    )
)

# Cost per Won Opportunity — right axis
fig_business.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["cost_per_won_opportunity"],
        name="Cost per Won Opportunity",
        mode="lines+markers",
        line={
            "width": 3,
            "color": cost_per_won_opportunity_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": cost_per_won_opportunity_color,
        },
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Cost per Won Opportunity: €%{y:,.0f}"
            "<extra></extra>"
        ),
        yaxis="y2",
    )
)

# ------------------------------------------------------------
# Paid ROAS — hidden third axis with visible labels
# ------------------------------------------------------------
fig_business.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["paid_roas"],
        name="Paid ROAS",
        mode="lines+markers+text",
        line={
            "width": 3,
            "dash": "dash",
            "color": paid_roas_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": paid_roas_color,
        },
        text=[
            f"{value:.2f}x"
            for value in performance_trends["paid_roas"]
        ],
        textposition="top center",
        textfont={
            "size": 11,
            "color": paid_roas_color,
        },
        cliponaxis=False,
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Paid ROAS: %{y:.2f}x"
            "<extra></extra>"
        ),
        yaxis="y3",
    )
)


style_xaxis(fig_business)

fig_business.update_layout(
    yaxis={
        "title": "Won Revenue (€)",
        "tickprefix": "€",
        "tickformat": "~s",
        "showgrid": False,
        "showline": True,
        "linewidth": 1,
        "linecolor": "#BDBDBD",
        "ticks": "outside",
        "ticklen": 5,
        "tickwidth": 1,
        "tickcolor": "#BDBDBD",
        "zeroline": False,
    },

    yaxis2={
        "title": "Cost per Won Opportunity (€)",
        "tickprefix": "€",
        "tickformat": "~s",
        "overlaying": "y",
        "side": "right",
        "showgrid": False,
        "showline": True,
        "linewidth": 1,
        "linecolor": "#BDBDBD",
        "ticks": "outside",
        "ticklen": 5,
        "tickwidth": 1,
        "tickcolor": "#BDBDBD",
        "zeroline": False,
    },

    yaxis3={
        "overlaying": "y",
        "side": "right",
        "anchor": "free",
        "position": 0.95,
        "showgrid": False,
        "showline": False,
        "showticklabels": False,
        "ticks": "",
        "zeroline": False,
    },
)

business_layout = deepcopy(CHART_STYLE)

business_layout["title"]["text"] = "Acquisition Cohort Performance"

business_layout["margin"] = {
    "l": 60,
    "r": 90,
    "t": 100,
    "b": 60,
}

business_layout["legend"] = {
    **business_layout.get("legend", {}),
    "orientation": "h",
    "x": 0.5,
    "xanchor": "center",
    "y": 1.10,
    "yanchor": "top",
}

fig_business.update_layout(
    **business_layout,
    height=500,
    hovermode="x unified",
)

fig_business.show()


# ------------------------------------------------------------
# 3.6 Funnel Performance
# Lead → Opportunity + Opportunity → Customer
# ------------------------------------------------------------

fig_funnel = go.Figure()

fig_funnel.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["lead_to_opportunity_rate"],
        name="Lead → Opportunity Rate",
        mode="lines+markers",
        line={
            "width": 3,
            "color": lead_to_opportunity_rate_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": lead_to_opportunity_rate_color,
        },
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Lead → Opportunity Rate: %{y:.1%}"
            "<extra></extra>"
        ),
    )
)

fig_funnel.add_trace(
    go.Scatter(
        x=performance_trends["month"],
        y=performance_trends["opportunity_win_rate"],
        name="Opportunity → Customer Rate",
        mode="lines+markers",
        line={
            "width": 3,
            "color": opportunity_win_rate_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        marker={
            "size": 7,
            "color": opportunity_win_rate_color,
        },
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Opportunity → Customer Rate: %{y:.1%}"
            "<extra></extra>"
        ),
    )
)

style_xaxis(fig_funnel)

fig_funnel.update_yaxes(
    title_text="Conversion Rate",
    tickformat=".0%",
    range=[0, 1],
)

style_layout(
    fig_funnel,
    "Acquisition Cohort Conversion",
)

fig_funnel.show()

##### Key Findings

- Marketing investment remained relatively stable across acquisition cohorts, providing a stable baseline for evaluating differences in downstream performance. Despite this, won revenue and Paid ROAS varied considerably across cohorts. Opportunity Win Rate exhibited a similar pattern, suggesting that cohorts with stronger opportunity conversion generally generated higher won revenue and marketing efficiency.

- Pipeline value remained relatively consistent across cohorts, indicating that similar levels of pipeline were generated from acquired leads. While the Lead → Opportunity Rate remained relatively stable, Opportunity Win Rate showed greater variation, highlighting the sales stage as the largest source of variation in funnel performance. This suggests that differences in Opportunity Win Rate were more closely associated with differences in won revenue than the Lead → Opportunity Rate during the reporting period.

- *Note: Marketing campaign data is available through September 2025, while CRM outcomes and product usage extend beyond this period. Marketing and downstream CRM outcomes are attributed to the lead acquisition period; therefore, the trends represent acquisition cohort performance rather than calendar-period revenue performance.*

## Step 4: Acquisition Performance
- Compare channels first, then campaigns, to identify which acquisition efforts generate the strongest commercial value.

In [4]:
# ------------------------------------------------------------
# 1. QUERY CHANNEL AND CAMPAIGN PERFORMANCE
# Ratios are recalculated from aggregated totals rather than
# averaging row-level ratios.
# ------------------------------------------------------------

channel_performance = conn.execute("""
    SELECT
        channel,

        SUM(total_ad_spend) AS total_ad_spend,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(total_leads) AS total_leads,
        SUM(total_opportunities) AS total_opportunities,
        SUM(won_opportunities) AS won_opportunities,

        SUM(total_pipeline_value) AS total_pipeline_value,
        SUM(total_won_revenue) AS total_won_revenue,

        SUM(total_won_revenue)
            / NULLIF(SUM(total_ad_spend), 0) AS paid_roas,

        SUM(total_ad_spend)
            / NULLIF(SUM(won_opportunities), 0) AS cost_per_won_opportunity,

        SUM(total_ad_spend)
            / NULLIF(SUM(total_leads), 0) AS cost_per_lead,

        SUM(clicks)
            / NULLIF(SUM(impressions), 0) AS click_through_rate,

        SUM(total_ad_spend)
            / NULLIF(SUM(clicks), 0) AS cost_per_click,

        SUM(total_won_revenue)
            / NULLIF(SUM(won_opportunities), 0) AS average_won_deal_value,

        SUM(average_sales_cycle_days * won_opportunities)
            / NULLIF(SUM(won_opportunities), 0) AS average_sales_cycle_days

    FROM mart_campaign_performance
    GROUP BY channel
    ORDER BY total_won_revenue DESC
""").df()


campaign_performance = conn.execute("""
    SELECT
        channel,
        campaign_name,

        SUM(total_ad_spend) AS total_ad_spend,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks,
        SUM(total_leads) AS total_leads,
        SUM(total_opportunities) AS total_opportunities,
        SUM(won_opportunities) AS won_opportunities,

        SUM(total_pipeline_value) AS total_pipeline_value,
        SUM(total_won_revenue) AS total_won_revenue,

        SUM(total_won_revenue)
            / NULLIF(SUM(total_ad_spend), 0) AS paid_roas,

        SUM(total_ad_spend)
            / NULLIF(SUM(won_opportunities), 0) AS cost_per_won_opportunity,

        SUM(total_ad_spend)
            / NULLIF(SUM(total_leads), 0) AS cost_per_lead,

        SUM(clicks)
            / NULLIF(SUM(impressions), 0) AS click_through_rate,

        SUM(total_ad_spend)
            / NULLIF(SUM(clicks), 0) AS cost_per_click,

        SUM(total_won_revenue)
            / NULLIF(SUM(won_opportunities), 0) AS average_won_deal_value,

        SUM(average_sales_cycle_days * won_opportunities)
            / NULLIF(SUM(won_opportunities), 0) AS average_sales_cycle_days

    FROM mart_campaign_performance
    GROUP BY
        channel,
        campaign_name
    ORDER BY total_won_revenue DESC
""").df()


# Replace infinite values produced by zero denominators
channel_performance = channel_performance.replace(
    [np.inf, -np.inf],
    np.nan,
)

campaign_performance = campaign_performance.replace(
    [np.inf, -np.inf],
    np.nan,
)

# More readable campaign label for charts
campaign_performance["campaign_label"] = (
    campaign_performance["campaign_name"]
    + " · "
    + campaign_performance["channel"]
)


# ------------------------------------------------------------
# 2. SHARED CHART FORMATTING
# ------------------------------------------------------------

CHART_STYLE = {
    "font": {
        "family": "Calibri",
        "size": 14,
        "color": "#2F2F2F",
    },
    "title": {
        "font": {
            "family": "Calibri",
            "size": 22,
            "color": "#2F2F2F",
        },
        "x": 0.5,
        "xanchor": "center",
        "y": 0.97,
        "yanchor": "top",
    },
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "margin": {
        "l": 40,
        "r": 40,
        "t": 80,
        "b": 40,
    },
    "hoverlabel": {
        "font": {
            "family": "Calibri",
            "size": 13,
        }
    },
    "legend": {
        "font": {
            "family": "Calibri",
            "size": 13,
        },
        "title": {
            "font": {
                "family": "Calibri",
                "size": 14,
            }
        },
    },
}


def apply_chart_style(fig, title, height=520):
    """Apply the common notebook chart design."""

    layout = deepcopy(CHART_STYLE)
    layout["title"]["text"] = f"<b>{title}</b>"
    layout["height"] = height

    fig.update_layout(**layout)

    fig.update_xaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    fig.update_yaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    return fig


# ============================================================
# VISUAL 1: CHANNEL COMMERCIAL VALUE
# Spend versus won revenue, with pipeline value represented
# by bubble size. Bubble colour identifies the channel, while
# Paid ROAS is displayed directly with each bubble.
# ============================================================

# Create a readable chart label with the channel and Paid ROAS
channel_performance["bubble_label"] = (
    channel_performance["channel"]
    + "<br>"
    + "ROAS: "
    + channel_performance["paid_roas"].map(lambda x: f"{x:.2f}x")
)

fig_channel_value = px.scatter(
    channel_performance,
    x="total_ad_spend",
    y="total_won_revenue",
    size="total_pipeline_value",
    color="channel",
    color_discrete_map={
        "Paid Search": "#EF553B",
        "Paid Social": "#636EFA",
        "Display": "#00CC96",
    },
    text="bubble_label",
    custom_data=[
        "channel",
        "total_pipeline_value",
        "paid_roas",
        "cost_per_won_opportunity",
        "cost_per_lead",
        "average_won_deal_value",
        "average_sales_cycle_days",
    ],
    labels={
        "total_ad_spend": "Total Ad Spend",
        "total_won_revenue": "Won Revenue",
        "channel": "Channel",
    },
    size_max=65,
)

fig_channel_value.update_traces(
    textposition="top center",
    cliponaxis=False,
    textfont={
        "family": "Calibri",
        "size": 13,
        "color": "#2F2F2F",
    },
    marker={
        "line": {
            "width": 1,
            "color": "white",
        }
    },
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Ad Spend: €%{x:,.0f}<br>"
        "Won Revenue: €%{y:,.0f}<br>"
        "Pipeline Value: €%{customdata[1]:,.0f}<br>"
        "Paid ROAS: %{customdata[2]:.2f}x<br>"
        "Cost per Won Opportunity: €%{customdata[3]:,.0f}<br>"
        "Cost per Lead: €%{customdata[4]:,.0f}<br>"
        "Average Won Deal Value: €%{customdata[5]:,.0f}<br>"
        "Average Sales Cycle: %{customdata[6]:.1f} days"
        "<extra></extra>"
    ),
)

fig_channel_value.update_xaxes(
    title_text="<b>Total Ad Spend</b>",
    tickprefix="€",
    tickformat="~s",
)

fig_channel_value.update_yaxes(
    title_text="<b>Won Revenue</b>",
    tickprefix="€",
    tickformat="~s",
)

apply_chart_style(
    fig_channel_value,
    "Channel Investment and Revenue Performance",
    height=560,
)

fig_channel_value.update_layout(
    showlegend=False
)

fig_channel_value.show()


# ============================================================
# VISUAL 2: CHANNEL EFFICIENCY
# Compare acquisition cost and marketing efficiency by channel.
# Cost per Lead and Cost per Won Opportunity share the primary
# axis; Paid ROAS is displayed on the secondary axis.
# ============================================================

channel_efficiency = channel_performance.sort_values(
    "paid_roas",
    ascending=False,
).copy()

fig_channel_efficiency = make_subplots(
    specs=[[{"secondary_y": True}]]
)

fig_channel_efficiency.add_trace(
    go.Bar(
        x=channel_efficiency["channel"],
        y=channel_efficiency["cost_per_lead"],
        name="Cost per Lead",
        text=channel_efficiency["cost_per_lead"],
        texttemplate="€%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Cost per Lead: €%{y:,.2f}"
            "<extra></extra>"
        ),
    ),
    secondary_y=False,
)

fig_channel_efficiency.add_trace(
    go.Bar(
        x=channel_efficiency["channel"],
        y=channel_efficiency["cost_per_won_opportunity"],
        name="Cost per Won Opportunity",
        text=channel_efficiency["cost_per_won_opportunity"],
        texttemplate="€%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Cost per Won Opportunity: €%{y:,.2f}"
            "<extra></extra>"
        ),
    ),
    secondary_y=False,
)

fig_channel_efficiency.add_trace(
    go.Scatter(
        x=channel_efficiency["channel"],
        y=channel_efficiency["paid_roas"],
        name="Paid ROAS",
        mode="lines+markers+text",
        text=channel_efficiency["paid_roas"],
        texttemplate="%{text:.2f}x",
        textposition="top center",
        marker={
            "size": 9,
            "color": paid_roas_color,
        },
        line={
            "width": 3,
            "dash": "dash",
            "color": paid_roas_color,
            "shape": "spline",
            "smoothing": 0.8,
        },
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Paid ROAS: %{y:.2f}x"
            "<extra></extra>"
        ),
    ),
    secondary_y=True,
)

fig_channel_efficiency.update_layout(
    barmode="group",
    bargap=0.25,
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.03,
        "xanchor": "center",
        "x": 0.5,
    },
)

fig_channel_efficiency.update_xaxes(
    title_text=None,
    categoryorder="array",
    categoryarray=channel_efficiency["channel"].tolist(),
)

fig_channel_efficiency.update_yaxes(
    title_text="<b>Acquisition Cost</b>",
    tickprefix="€",
    tickformat="~s",
    secondary_y=False,
)

fig_channel_efficiency.update_yaxes(
    title_text="<b>Paid ROAS</b>",
    tickformat=".1f",
    ticksuffix="x",
    rangemode="tozero",
    secondary_y=True,
)

apply_chart_style(
    fig_channel_efficiency,
    "Channel Acquisition Efficiency",
    height=560,
)

fig_channel_efficiency.show()


# ============================================================
# VISUAL 3: CAMPAIGN INVESTMENT AND REVENUE PERFORMANCE
# Campaign names are displayed directly on the bubbles so that
# key campaigns can be identified without relying on hover.
# ============================================================

# Create a shorter label for the chart while preserving the
# full campaign name in the hover tooltip
campaign_performance["campaign_chart_label"] = (
    campaign_performance["campaign_name"]
    .str.replace("Campaign", "", regex=False)
    .str.strip()
    .str.slice(0, 20)
)

fig_campaign_value = px.scatter(
    campaign_performance,
    x="total_ad_spend",
    y="total_won_revenue",
    size="total_pipeline_value",
    color="channel",
    text="campaign_chart_label",
    custom_data=[
        "campaign_name",
        "channel",
        "paid_roas",
        "total_pipeline_value",
        "cost_per_won_opportunity",
        "cost_per_lead",
        "click_through_rate",
        "cost_per_click",
        "average_won_deal_value",
        "average_sales_cycle_days",
    ],
    labels={
        "total_ad_spend": "Total Ad Spend",
        "total_won_revenue": "Won Revenue",
        "channel": "Channel",
    },
    size_max=55,
)

fig_campaign_value.update_traces(
    textposition="top center",
    cliponaxis=False,
    textfont=dict(
        family="Calibri",
        size=10,
        color="#2F2F2F"
    ),
    marker=dict(
        line=dict(
            width=1,
            color="white"
        )
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Channel: %{customdata[1]}<br>"
        "Ad Spend: €%{x:,.0f}<br>"
        "Won Revenue: €%{y:,.0f}<br>"
        "Paid ROAS: %{customdata[2]:.2f}x<br>"
        "Pipeline Value: €%{customdata[3]:,.0f}<br>"
        "Cost per Won Opportunity: €%{customdata[4]:,.0f}<br>"
        "Cost per Lead: €%{customdata[5]:,.0f}<br>"
        "CTR: %{customdata[6]:.2%}<br>"
        "CPC: €%{customdata[7]:.2f}<br>"
        "Average Won Deal Value: €%{customdata[8]:,.0f}<br>"
        "Average Sales Cycle: %{customdata[9]:.1f} days"
        "<extra></extra>"
    ),
)

fig_campaign_value.update_xaxes(
    title_text="<b>Total Ad Spend</b>",
    tickprefix="€",
    tickformat="~s",
)

fig_campaign_value.update_yaxes(
    title_text="<b>Won Revenue</b>",
    tickprefix="€",
    tickformat="~s",
)

fig_campaign_value.update_layout(
    legend={
        "title": {
            "text": "<b>Channel</b>",
        },
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "center",
        "x": 0.5,
    }
)

apply_chart_style(
    fig_campaign_value,
    "Campaign Investment and Revenue Performance",
    height=650,
)

# Add slightly more space to accommodate campaign labels
fig_campaign_value.update_layout(
    margin={
        "l": 20,
        "r": 20,
        "t": 95,
        "b": 50,
    }
)

fig_campaign_value.show()


# ============================================================
# VISUAL 4: TOP AND BOTTOM CAMPAIGNS BY PAID ROAS
# Compare the five strongest and five weakest campaigns by
# Paid ROAS, alongside acquisition cost, deal value and
# average sales-cycle length.
# ============================================================

# Remove campaigns without a valid ROAS before ranking
campaign_ranking = (
    campaign_performance
    .dropna(subset=["paid_roas"])
    .copy()
)

# Select the five highest-ROAS campaigns
top_5_campaigns = (
    campaign_ranking
    .nlargest(5, "paid_roas")
    .assign(performance_group="Top 5")
)

# Exclude the top campaigns before selecting the bottom five
# to prevent overlap when fewer than ten campaigns are available
top_campaign_labels = set(
    top_5_campaigns["campaign_label"]
)

bottom_5_campaigns = (
    campaign_ranking[
        ~campaign_ranking["campaign_label"].isin(top_campaign_labels)
    ]
    .nsmallest(5, "paid_roas")
    .assign(performance_group="Bottom 5")
)

# Combine the bottom and top campaigns
campaign_matrix = pd.concat(
    [
        bottom_5_campaigns,
        top_5_campaigns,
    ],
    ignore_index=True,
)

# Sort campaigns from lowest to highest ROAS so the strongest
# campaigns appear at the top of the horizontal chart
campaign_matrix = (
    campaign_matrix
    .sort_values(
        "paid_roas",
        ascending=True,
    )
    .copy()
)

# Use only the campaign label on the visible y-axis.
# The ranking group remains available in the hover tooltip.
campaign_matrix["campaign_comparison_label"] = (
    campaign_matrix["campaign_label"]
)


fig_campaign_matrix = make_subplots(
    rows=1,
    cols=4,
    shared_yaxes=True,
    horizontal_spacing=0.055,
    subplot_titles=[
        "<b>Paid ROAS</b>",
        "<b>Cost per Won Opportunity</b>",
        "<b>Average Won Deal Value</b>",
        "<b>Average Sales Cycle</b>",
    ],
)


# ------------------------------------------------------------
# Paid ROAS
# ------------------------------------------------------------

fig_campaign_matrix.add_trace(
    go.Bar(
        x=campaign_matrix["paid_roas"],
        y=campaign_matrix["campaign_comparison_label"],
        orientation="h",
        text=campaign_matrix["paid_roas"],
        texttemplate="%{text:.2f}x",
        textposition="outside",
        cliponaxis=False,
        name="Paid ROAS",
        customdata=campaign_matrix[
            [
                "campaign_name",
                "channel",
                "performance_group",
                "total_ad_spend",
                "total_won_revenue",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Channel: %{customdata[1]}<br>"
            "Ranking Group: %{customdata[2]}<br>"
            "Paid ROAS: %{x:.2f}x<br>"
            "Ad Spend: €%{customdata[3]:,.0f}<br>"
            "Won Revenue: €%{customdata[4]:,.0f}"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)

# ------------------------------------------------------------
# Cost per Won Opportunity
# ------------------------------------------------------------

fig_campaign_matrix.add_trace(
    go.Bar(
        x=campaign_matrix["cost_per_won_opportunity"],
        y=campaign_matrix["campaign_comparison_label"],
        orientation="h",
        text=campaign_matrix["cost_per_won_opportunity"],
        texttemplate="€%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        name="Cost per Won Opportunity",
        customdata=campaign_matrix[
            [
                "campaign_name",
                "channel",
                "performance_group",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Channel: %{customdata[1]}<br>"
            "Ranking Group: %{customdata[2]}<br>"
            "Cost per Won Opportunity: €%{x:,.2f}"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=2,
)


# ------------------------------------------------------------
# Average Won Deal Value
# ------------------------------------------------------------

fig_campaign_matrix.add_trace(
    go.Bar(
        x=campaign_matrix["average_won_deal_value"],
        y=campaign_matrix["campaign_comparison_label"],
        orientation="h",
        text=campaign_matrix["average_won_deal_value"],
        texttemplate="€%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        name="Average Won Deal Value",
        customdata=campaign_matrix[
            [
                "campaign_name",
                "channel",
                "performance_group",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Channel: %{customdata[1]}<br>"
            "Ranking Group: %{customdata[2]}<br>"
            "Average Won Deal Value: €%{x:,.0f}"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=3,
)


# ------------------------------------------------------------
# Average Sales Cycle
# ------------------------------------------------------------

fig_campaign_matrix.add_trace(
    go.Bar(
        x=campaign_matrix["average_sales_cycle_days"],
        y=campaign_matrix["campaign_comparison_label"],
        orientation="h",
        text=campaign_matrix["average_sales_cycle_days"],
        texttemplate="%{text:.0f}d",
        textposition="outside",
        cliponaxis=False,
        name="Average Sales Cycle",
        customdata=campaign_matrix[
            [
                "campaign_name",
                "channel",
                "performance_group",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Channel: %{customdata[1]}<br>"
            "Ranking Group: %{customdata[2]}<br>"
            "Average Sales Cycle: %{x:.1f} days"
            "<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=4,
)


# ------------------------------------------------------------
# Axis formatting
# ------------------------------------------------------------

fig_campaign_matrix.update_xaxes(
    title_text="<b>ROAS</b>",
    tickformat=".1f",
    ticksuffix="x",
    rangemode="tozero",
    row=1,
    col=1,
)

fig_campaign_matrix.update_xaxes(
    title_text="<b>Acquisition Cost</b>",
    tickprefix="€",
    tickformat="~s",
    rangemode="tozero",
    row=1,
    col=2,
)

fig_campaign_matrix.update_xaxes(
    title_text="<b>Deal Value</b>",
    tickprefix="€",
    tickformat="~s",
    rangemode="tozero",
    row=1,
    col=3,
)

fig_campaign_matrix.update_xaxes(
    title_text="<b>Days</b>",
    tickformat=".0f",
    rangemode="tozero",
    row=1,
    col=4,
)

fig_campaign_matrix.update_yaxes(
    title_text=None,
    automargin=True,
    row=1,
    col=1,
)

# Hide repeated campaign labels on the remaining panels
for column in [2, 3, 4]:
    fig_campaign_matrix.update_yaxes(
        showticklabels=False,
        row=1,
        col=column,
    )


# ------------------------------------------------------------
# Apply shared styling
# ------------------------------------------------------------

apply_chart_style(
    fig_campaign_matrix,
    "Campaign Performance Comparison: Top and Bottom by Paid ROAS",
    height=max(620, 120 + len(campaign_matrix) * 48),
)

fig_campaign_matrix.update_layout(
    margin={
        "l": 80,
        "r": 60,
        "t": 100,
        "b": 50,
    }
)

fig_campaign_matrix.show()


# ------------------------------------------------------------
# OPTIONAL: DISPLAY THE UNDERLYING SUMMARY TABLES
# ------------------------------------------------------------

display(
    channel_performance[
        [
            "channel",
            "total_ad_spend",
            "total_won_revenue",
            "total_pipeline_value",
            "paid_roas",
            "cost_per_won_opportunity",
            "cost_per_lead",
            "click_through_rate",
            "cost_per_click",
            "average_won_deal_value",
            "average_sales_cycle_days",
        ]
    ].sort_values("paid_roas", ascending=False)
)

display(
    campaign_performance[
        [
            "channel",
            "campaign_name",
            "total_ad_spend",
            "total_won_revenue",
            "total_pipeline_value",
            "paid_roas",
            "cost_per_won_opportunity",
            "cost_per_lead",
            "click_through_rate",
            "cost_per_click",
            "average_won_deal_value",
            "average_sales_cycle_days",
        ]
    ].sort_values("paid_roas", ascending=False)
)

,channel,total_ad_spend,total_won_revenue,total_pipeline_value,paid_roas,cost_per_won_opportunity,cost_per_lead,click_through_rate,cost_per_click,average_won_deal_value,average_sales_cycle_days
1,Display,88253.38,584006.0,980840.0,6.617378,3043.220000,519.137529,0.007929,1.649843,20138.137931,51.551724
0,Paid Social,1018534.81,673889.0,1770485.0,0.661626,31829.212813,4025.829289,0.019990,5.491287,21059.031250,52.000000
2,Paid Search,451084.80,132784.0,308550.0,0.294366,64440.685714,10251.927273,0.039300,4.055278,18969.142857,57.714286


,channel,campaign_name,total_ad_spend,total_won_revenue,total_pipeline_value,paid_roas,cost_per_won_opportunity,cost_per_lead,click_through_rate,cost_per_click,average_won_deal_value,average_sales_cycle_days
1,Display,APAC_Brand_LinkedIn,14624.73,136254.0,208053.0,9.316685,2437.455000,487.491000,0.008030,1.636425,22709.000000,62.833333
2,Display,LATAM_Competitor_Display,14659.74,134240.0,171514.0,9.157052,2443.290000,505.508276,0.007923,1.637594,22373.333333,64.333333
3,Display,APAC_Competitor_Search,14736.88,114974.0,238290.0,7.801787,2456.146667,526.317143,0.007854,1.669145,19162.333333,50.166667
4,Display,LATAM_FeatureX_Display,14675.97,110167.0,139924.0,7.506625,2096.567143,524.141786,0.007971,1.634113,15738.142857,48.571429
7,Display,US_Brand_LinkedIn,14406.66,69346.0,128763.0,4.813468,4802.220000,411.618857,0.007715,1.651950,23115.333333,26.666667
16,Display,EMEA_Brand_Search,15149.40,19025.0,94296.0,1.255825,15149.400000,757.470000,0.008082,1.669907,19025.000000,11.000000
0,Paid Social,US_FeatureX_Search,191010.06,200762.0,516077.0,1.051055,21223.340000,3673.270385,0.020260,5.514625,22306.888889,50.777778
6,Paid Social,APAC_Brand_Search,87523.44,75208.0,136343.0,0.859290,29174.480000,4167.782857,0.019142,5.500468,25069.333333,68.333333
8,Paid Social,LATAM_Brand_Display,91943.64,68844.0,148612.0,0.748763,22985.910000,3997.549565,0.019490,5.669933,17211.000000,50.250000
9,Paid Social,LATAM_Brand_LinkedIn,93936.42,66870.0,136673.0,0.711864,31312.140000,4473.162857,0.020455,5.448116,22290.000000,76.000000


### Key Findings

- **Display** delivered the strongest marketing efficiency despite receiving the smallest marketing investment (**€88K**). It generated **€584K in won revenue** from a **€981K pipeline**, achieving the highest **Paid ROAS (6.62x)** together with the lowest **Cost per Lead (€519)** and **Cost per Won Opportunity (€3.0K)**. In contrast, **Paid Search** generated only **€133K in won revenue** from **€451K** in ad spend, resulting in the lowest **Paid ROAS (0.29x)** and the highest acquisition costs.

- **Paid Social** accounted for the largest marketing investment (**€1.02M**) and generated the highest **won revenue (€674K)** and **pipeline value (€1.77M)**. However, its **Paid ROAS of 0.66x** indicates that higher investment did not translate into proportionally higher attributed returns, highlighting an opportunity to improve campaign efficiency within the channel.

- Campaign performance varied substantially within channels. **Display campaigns** consistently dominated the top-performing rankings, with the five highest campaigns achieving **Paid ROAS between 4.81x and 9.32x** while maintaining **Cost per Won Opportunity below €4.9K**. This demonstrates that Display's strong channel performance was driven by consistently efficient campaigns rather than a single outlier.

- Conversely, the weakest-performing campaigns were concentrated within **Paid Search** and **Paid Social**, with **Paid ROAS ranging from 0.14x to 0.48x** and **Cost per Won Opportunity reaching €113K**. While average won deal values remained relatively similar across campaigns, the substantial differences in marketing efficiency suggest that campaign execution and acquisition effectiveness varied considerably, warranting further investigation in the funnel conversion analysis.

## Step 5: Acquisition Conversion Performance
- Where are we losing potential customers?
- The aim is to Compare lead progression and conversion performance across channels and campaigns to identify the largest acquisition bottlenecks.

In [5]:
# ------------------------------------------------------------
# 5.1 Load acquisition conversion data
# ------------------------------------------------------------

conversion_data = conn.execute(
    """
    SELECT
        campaign_id,
        campaign_name,
        channel,
        total_leads,
        total_opportunities,
        won_opportunities
    FROM mart_campaign_performance
    """
).df()


# Ensure volume metrics are numeric
volume_columns = [
    "total_leads",
    "total_opportunities",
    "won_opportunities",
]

conversion_data[volume_columns] = (
    conversion_data[volume_columns]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)


# Replace missing category values
conversion_data["channel"] = (
    conversion_data["channel"]
    .fillna("Unknown Channel")
)

conversion_data["campaign_name"] = (
    conversion_data["campaign_name"]
    .fillna("Unknown Campaign")
)


# ------------------------------------------------------------
# Helper function: safe division
# ------------------------------------------------------------

def safe_divide(numerator, denominator):
    """
    Divide two pandas Series and return NaN where the
    denominator is zero.
    """
    return np.where(
        denominator > 0,
        numerator / denominator,
        np.nan,
    )


# ------------------------------------------------------------
# Helper function: shared chart styling
# ------------------------------------------------------------

def style_conversion_chart(fig, title, height=550):
    """
    Apply the shared notebook Plotly style.
    """
    layout = deepcopy(CHART_STYLE)

    layout["title"]["text"] = f"<b>{title}</b>"
    layout["height"] = height

    fig.update_layout(**layout)

    fig.update_xaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    fig.update_yaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    return fig


# ============================================================
# 5.2 CHANNEL-LEVEL CONVERSION PERFORMANCE
# ============================================================

channel_conversion = (
    conversion_data
    .groupby(
        "channel",
        as_index=False,
        dropna=False,
    )
    .agg(
        total_leads=("total_leads", "sum"),
        total_opportunities=("total_opportunities", "sum"),
        won_opportunities=("won_opportunities", "sum"),
    )
)


# Recalculate conversion rates from aggregated totals
channel_conversion["lead_to_opportunity_rate"] = safe_divide(
    channel_conversion["total_opportunities"],
    channel_conversion["total_leads"],
)

channel_conversion["opportunity_win_rate"] = safe_divide(
    channel_conversion["won_opportunities"],
    channel_conversion["total_opportunities"],
)

channel_conversion["lead_to_won_rate"] = safe_divide(
    channel_conversion["won_opportunities"],
    channel_conversion["total_leads"],
)


# Calculate absolute losses
channel_conversion["leads_not_converted"] = (
    channel_conversion["total_leads"]
    - channel_conversion["total_opportunities"]
)

channel_conversion["opportunities_not_won"] = (
    channel_conversion["total_opportunities"]
    - channel_conversion["won_opportunities"]
)


# Calculate loss rates
channel_conversion["lead_to_opportunity_loss_rate"] = (
    1 - channel_conversion["lead_to_opportunity_rate"]
)

channel_conversion["opportunity_loss_rate"] = (
    1 - channel_conversion["opportunity_win_rate"]
)


# Identify the weaker conversion stage
channel_conversion["largest_bottleneck"] = np.where(
    channel_conversion["lead_to_opportunity_rate"]
    <= channel_conversion["opportunity_win_rate"],
    "Lead → Opportunity",
    "Opportunity → Won",
)


# Sort channels by lead volume
channel_conversion = (
    channel_conversion
    .sort_values(
        "total_leads",
        ascending=False,
    )
    .reset_index(drop=True)
)


# Plotly horizontal category axes are displayed from bottom to top.
# Reversing this list places the highest-volume channel at the top.
channel_order = (
    channel_conversion["channel"]
    .tolist()[::-1]
)


# ============================================================
# VISUAL 1: CHANNEL CONVERSION VOLUME
#
# Horizontal grouped bars:
# - Company count on the x-axis
# - Channels on the y-axis
#
# Traces are added in reverse funnel order because Plotly places
# the first horizontal grouped-bar trace lower within each group.
#
# Final visual order:
# Leads
# Opportunities
# Won Opportunities
# ============================================================

fig_channel_volume = go.Figure()


# Add the final stage first so it appears at the bottom
fig_channel_volume.add_trace(
    go.Bar(
        x=channel_conversion["won_opportunities"],
        y=channel_conversion["channel"],
        orientation="h",
        name="Won Opportunities",
        marker_color="#19D3F3",  # Cyan
        text=channel_conversion["won_opportunities"],
        texttemplate="%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        customdata=channel_conversion[
            [
                "total_leads",
                "total_opportunities",
                "lead_to_won_rate",
            ]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Stage: Won Opportunities<br>"
            "Companies: %{x:,.0f}<br>"
            "Total Leads: %{customdata[0]:,.0f}<br>"
            "Total Opportunities: %{customdata[1]:,.0f}<br>"
            "Lead → Won Rate: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    )
)


# Add the middle stage second
fig_channel_volume.add_trace(
    go.Bar(
        x=channel_conversion["total_opportunities"],
        y=channel_conversion["channel"],
        orientation="h",
        name="Opportunities",
        marker_color="#FF6692",  # Pink / magenta
        text=channel_conversion["total_opportunities"],
        texttemplate="%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        customdata=channel_conversion[
            [
                "total_leads",
                "won_opportunities",
                "lead_to_opportunity_rate",
            ]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Stage: Opportunities<br>"
            "Companies: %{x:,.0f}<br>"
            "Total Leads: %{customdata[0]:,.0f}<br>"
            "Won Opportunities: %{customdata[1]:,.0f}<br>"
            "Lead → Opportunity Rate: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    )
)


# Add the first stage last so it appears at the top
fig_channel_volume.add_trace(
    go.Bar(
        x=channel_conversion["total_leads"],
        y=channel_conversion["channel"],
        orientation="h",
        name="Leads",
        marker_color="#636EFA",  # Blue
        text=channel_conversion["total_leads"],
        texttemplate="%{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        customdata=channel_conversion[
            [
                "total_opportunities",
                "won_opportunities",
                "lead_to_won_rate",
            ]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Stage: Leads<br>"
            "Companies: %{x:,.0f}<br>"
            "Total Opportunities: %{customdata[0]:,.0f}<br>"
            "Won Opportunities: %{customdata[1]:,.0f}<br>"
            "Lead → Won Rate: %{customdata[2]:.1%}"
            "<extra></extra>"
        ),
    )
)


fig_channel_volume.update_layout(
    barmode="group",
    bargap=0.25,
    bargroupgap=0.08,
    legend=dict(
        title=dict(
            text="<b>Stage</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,

        # Reverse the legend so it reads:
        # Leads → Opportunities → Won Opportunities
        traceorder="reversed",
    ),
    margin=dict(
        l=130,
        r=80,
        t=105,
        b=50,
    ),
)


fig_channel_volume.update_xaxes(
    title_text="<b>Company Count</b>",
    tickformat="~s",
    rangemode="tozero",
)


fig_channel_volume.update_yaxes(
    title_text=None,
    categoryorder="array",
    categoryarray=channel_order,
)


style_conversion_chart(
    fig_channel_volume,
    "Channel Conversion Volume",
    height=520,
)


fig_channel_volume.show()


# ============================================================
# VISUAL 2: CHANNEL CONVERSION PERFORMANCE
#
# Horizontal grouped bars:
# - Conversion percentage on the x-axis
# - Channels on the y-axis
#
# Final visual order:
# Lead → Opportunity Rate
# Opportunity Win Rate
# ============================================================

fig_channel_rates = go.Figure()


# Add the later conversion stage first so it appears lower
fig_channel_rates.add_trace(
    go.Bar(
        x=channel_conversion["opportunity_win_rate"],
        y=channel_conversion["channel"],
        orientation="h",
        name="Opportunity Win Rate",
        marker_color="#19D3F3",  # Cyan
        text=channel_conversion["opportunity_win_rate"],
        texttemplate="%{text:.1%}",
        textposition="outside",
        cliponaxis=False,
        customdata=channel_conversion[
            [
                "total_leads",
                "total_opportunities",
                "won_opportunities",
                "largest_bottleneck",
            ]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Stage: Opportunity → Won<br>"
            "Opportunity Win Rate: %{x:.1%}<br>"
            "Total Leads: %{customdata[0]:,.0f}<br>"
            "Total Opportunities: %{customdata[1]:,.0f}<br>"
            "Won Opportunities: %{customdata[2]:,.0f}<br>"
            "Largest Bottleneck: %{customdata[3]}"
            "<extra></extra>"
        ),
    )
)


# Add the earlier conversion stage last so it appears higher
fig_channel_rates.add_trace(
    go.Bar(
        x=channel_conversion["lead_to_opportunity_rate"],
        y=channel_conversion["channel"],
        orientation="h",
        name="Lead → Opportunity Rate",
        marker_color="#FF6692",  # Pink / magenta
        text=channel_conversion["lead_to_opportunity_rate"],
        texttemplate="%{text:.1%}",
        textposition="outside",
        cliponaxis=False,
        customdata=channel_conversion[
            [
                "total_leads",
                "total_opportunities",
                "won_opportunities",
                "largest_bottleneck",
            ]
        ],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Stage: Lead → Opportunity<br>"
            "Lead → Opportunity Rate: %{x:.1%}<br>"
            "Total Leads: %{customdata[0]:,.0f}<br>"
            "Total Opportunities: %{customdata[1]:,.0f}<br>"
            "Won Opportunities: %{customdata[2]:,.0f}<br>"
            "Largest Bottleneck: %{customdata[3]}"
            "<extra></extra>"
        ),
    )
)


fig_channel_rates.update_layout(
    barmode="group",
    bargap=0.25,
    bargroupgap=0.08,
    legend=dict(
        title=dict(
            text="<b>Conversion Stage</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,

        # Reverse the legend so it follows the conversion path
        traceorder="reversed",
    ),
    margin=dict(
        l=130,
        r=80,
        t=105,
        b=50,
    ),
)


fig_channel_rates.update_xaxes(
    title_text="<b>Conversion Rate</b>",
    tickformat=".0%",
    range=[0, 1],
)


fig_channel_rates.update_yaxes(
    title_text=None,
    categoryorder="array",
    categoryarray=channel_order,
)


style_conversion_chart(
    fig_channel_rates,
    "Channel Conversion Performance",
    height=520,
)


fig_channel_rates.show()


# ============================================================
# 5.3 CAMPAIGN-LEVEL CONVERSION PERFORMANCE
# ============================================================

campaign_conversion = (
    conversion_data
    .groupby(
        [
            "campaign_id",
            "campaign_name",
            "channel",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        total_leads=("total_leads", "sum"),
        total_opportunities=("total_opportunities", "sum"),
        won_opportunities=("won_opportunities", "sum"),
    )
)


# Recalculate campaign-level conversion rates
campaign_conversion["lead_to_opportunity_rate"] = safe_divide(
    campaign_conversion["total_opportunities"],
    campaign_conversion["total_leads"],
)

campaign_conversion["opportunity_win_rate"] = safe_divide(
    campaign_conversion["won_opportunities"],
    campaign_conversion["total_opportunities"],
)

campaign_conversion["lead_to_won_rate"] = safe_divide(
    campaign_conversion["won_opportunities"],
    campaign_conversion["total_leads"],
)


# Calculate absolute losses
campaign_conversion["leads_not_converted"] = (
    campaign_conversion["total_leads"]
    - campaign_conversion["total_opportunities"]
)

campaign_conversion["opportunities_not_won"] = (
    campaign_conversion["total_opportunities"]
    - campaign_conversion["won_opportunities"]
)


# Calculate loss rates
campaign_conversion["lead_to_opportunity_loss_rate"] = (
    1 - campaign_conversion["lead_to_opportunity_rate"]
)

campaign_conversion["opportunity_loss_rate"] = (
    1 - campaign_conversion["opportunity_win_rate"]
)


# Identify the weaker conversion stage
campaign_conversion["largest_bottleneck"] = np.where(
    campaign_conversion["lead_to_opportunity_rate"]
    <= campaign_conversion["opportunity_win_rate"],
    "Lead → Opportunity",
    "Opportunity → Won",
)


# Create readable campaign labels
campaign_conversion["campaign_label"] = (
    campaign_conversion["campaign_name"]
    .str.replace("Campaign", "", regex=False)
    .str.strip()
    .str.slice(0, 28)
)


# Keep campaigns with valid conversion denominators
campaign_conversion_valid = (
    campaign_conversion[
        (campaign_conversion["total_leads"] > 0)
        & (campaign_conversion["total_opportunities"] > 0)
        & campaign_conversion["lead_to_opportunity_rate"].notna()
        & campaign_conversion["opportunity_win_rate"].notna()
    ]
    .copy()
)


# Sort campaigns by overall Lead → Won Rate
campaign_conversion_valid = (
    campaign_conversion_valid
    .sort_values(
        [
            "lead_to_won_rate",
            "total_leads",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


# Lowest-performing campaigns appear at the bottom and
# highest-performing campaigns appear at the top.
campaign_order = (
    campaign_conversion_valid["campaign_label"]
    .tolist()[::-1]
)


# ============================================================
# VISUAL 3: CAMPAIGN CONVERSION PERFORMANCE
#
# Horizontal grouped bars:
# - Conversion percentage on the x-axis
# - Campaigns on the y-axis
#
# Final visual order within each campaign:
# Lead → Opportunity Rate
# Opportunity Win Rate
# ============================================================

fig_campaign_rates = go.Figure()


# Add Opportunity Win Rate first so it appears below
fig_campaign_rates.add_trace(
    go.Bar(
        x=campaign_conversion_valid["opportunity_win_rate"],
        y=campaign_conversion_valid["campaign_label"],
        orientation="h",
        name="Opportunity Win Rate",
        marker_color="#19D3F3",  # Cyan
        text=campaign_conversion_valid["opportunity_win_rate"],
        texttemplate="%{text:.0%}",
        textposition="outside",
        cliponaxis=False,
        customdata=campaign_conversion_valid[
            [
                "campaign_name",
                "channel",
                "total_leads",
                "total_opportunities",
                "won_opportunities",
                "lead_to_won_rate",
                "largest_bottleneck",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Channel: %{customdata[1]}<br>"
            "Stage: Opportunity → Won<br>"
            "Opportunity Win Rate: %{x:.1%}<br>"
            "Total Leads: %{customdata[2]:,.0f}<br>"
            "Total Opportunities: %{customdata[3]:,.0f}<br>"
            "Won Opportunities: %{customdata[4]:,.0f}<br>"
            "Lead → Won Rate: %{customdata[5]:.1%}<br>"
            "Largest Bottleneck: %{customdata[6]}"
            "<extra></extra>"
        ),
    )
)


# Add Lead → Opportunity Rate last so it appears above
fig_campaign_rates.add_trace(
    go.Bar(
        x=campaign_conversion_valid["lead_to_opportunity_rate"],
        y=campaign_conversion_valid["campaign_label"],
        orientation="h",
        name="Lead → Opportunity Rate",
        marker_color="#FF6692",  # Pink / magenta
        text=campaign_conversion_valid["lead_to_opportunity_rate"],
        texttemplate="%{text:.0%}",
        textposition="outside",
        cliponaxis=False,
        customdata=campaign_conversion_valid[
            [
                "campaign_name",
                "channel",
                "total_leads",
                "total_opportunities",
                "won_opportunities",
                "lead_to_won_rate",
                "largest_bottleneck",
            ]
        ],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Channel: %{customdata[1]}<br>"
            "Stage: Lead → Opportunity<br>"
            "Lead → Opportunity Rate: %{x:.1%}<br>"
            "Total Leads: %{customdata[2]:,.0f}<br>"
            "Total Opportunities: %{customdata[3]:,.0f}<br>"
            "Won Opportunities: %{customdata[4]:,.0f}<br>"
            "Lead → Won Rate: %{customdata[5]:.1%}<br>"
            "Largest Bottleneck: %{customdata[6]}"
            "<extra></extra>"
        ),
    )
)


fig_campaign_rates.update_layout(
    barmode="group",
    bargap=0.25,
    bargroupgap=0.08,
    legend=dict(
        title=dict(
            text="<b>Conversion Stage</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=0.99,
        xanchor="center",
        x=0.5,

        # Reverse the legend so the earlier stage appears first
        traceorder="reversed",
    ),
    margin=dict(
        l=220,
        r=80,
        t=110,
        b=55,
    ),
)


fig_campaign_rates.update_xaxes(
    title_text="<b>Conversion Rate</b>",
    tickformat=".0%",
    range=[0, 1],
)


fig_campaign_rates.update_yaxes(
    title_text=None,
    categoryorder="array",
    categoryarray=campaign_order,
    tickfont=dict(
        family="Calibri",
        size=11,
    ),
)


style_conversion_chart(
    fig_campaign_rates,
    "Campaign Conversion Performance",
    height=max(
        650,
        40 * len(campaign_conversion_valid),
    ),
)


fig_campaign_rates.show()


# ============================================================
# VISUAL 4: Campaign Conversion Rate Comparison
#
# X-axis: Lead → Opportunity Rate
# Y-axis: Opportunity Win Rate
# Bubble size: Total Leads
# Color: Channel
#
# Campaigns in the upper-right perform strongly at both stages.
# Campaigns in the lower-left perform weakly at both stages.
# ============================================================

fig_campaign_matrix = px.scatter(
    campaign_conversion_valid,
    x="lead_to_opportunity_rate",
    y="opportunity_win_rate",
    size="total_leads",
    color="channel",
 color_discrete_map={
    "Display": "#00CC96",      # Green
    "Paid Search": "#EF553B",  # Red-orange
    "Paid Social": "#636EFA",  # Blue
},
    text="campaign_label",
    size_max=50,
    custom_data=[
        "campaign_name",
        "channel",
        "total_leads",
        "total_opportunities",
        "won_opportunities",
        "lead_to_won_rate",
        "largest_bottleneck",
    ],
    labels={
        "lead_to_opportunity_rate":
            "Lead → Opportunity Rate",
        "opportunity_win_rate":
            "Opportunity Win Rate",
        "channel":
            "Channel",
        "total_leads":
            "Total Leads",
    },
)


fig_campaign_matrix.update_traces(
    textposition="top center",
    cliponaxis=False,
    textfont=dict(
        family="Calibri",
        size=9,
        color="#2F2F2F",
    ),
    marker=dict(
        line=dict(
            width=1,
            color="white",
        )
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Channel: %{customdata[1]}<br>"
        "Total Leads: %{customdata[2]:,.0f}<br>"
        "Total Opportunities: %{customdata[3]:,.0f}<br>"
        "Won Opportunities: %{customdata[4]:,.0f}<br>"
        "Lead → Opportunity Rate: %{x:.1%}<br>"
        "Opportunity Win Rate: %{y:.1%}<br>"
        "Lead → Won Rate: %{customdata[5]:.1%}<br>"
        "Largest Bottleneck: %{customdata[6]}"
        "<extra></extra>"
    ),
)


fig_campaign_matrix.update_xaxes(
    title_text="<b>Lead → Opportunity Rate</b>",
    tickformat=".0%",
    range=[0, 0.7],
)


fig_campaign_matrix.update_yaxes(
    title_text="<b>Opportunity Win Rate</b>",
    tickformat=".0%",
    range=[0, 1],
)


fig_campaign_matrix.update_layout(
    legend=dict(
        title=dict(
            text="<b>Channel</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=0.98,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(
        l=70,
        r=70,
        t=110,
        b=65,
    ),
)


style_conversion_chart(
    fig_campaign_matrix,
    "Campaign Conversion Rate Comparison",
    height=750,
)


fig_campaign_matrix.show()


# ============================================================
# 5.4 CHANNEL CONVERSION SUMMARY TABLE
# ============================================================

channel_conversion_summary = (
    channel_conversion[
        [
            "channel",
            "total_leads",
            "total_opportunities",
            "won_opportunities",
            "lead_to_opportunity_rate",
            "opportunity_win_rate",
            "lead_to_won_rate",
            "largest_bottleneck",
        ]
    ]
    .copy()
)


channel_conversion_summary.columns = [
    "Channel",
    "Total Leads",
    "Total Opportunities",
    "Won Opportunities",
    "Lead → Opportunity Rate",
    "Opportunity Win Rate",
    "Lead → Won Rate",
    "Largest Bottleneck",
]


channel_conversion_summary_display = (
    channel_conversion_summary
    .style
    .format(
        {
            "Total Leads": "{:,.0f}",
            "Total Opportunities": "{:,.0f}",
            "Won Opportunities": "{:,.0f}",
            "Lead → Opportunity Rate": "{:.1%}",
            "Opportunity Win Rate": "{:.1%}",
            "Lead → Won Rate": "{:.1%}",
        }
    )
    .set_properties(
        **{
            "font-family": "Calibri",
            "text-align": "center",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("font-family", "Calibri"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("background-color", "#F5F5F5"),
                ],
            },
            {
                "selector": "td:first-child",
                "props": [
                    ("text-align", "left"),
                ],
            },
        ]
    )
)


display(channel_conversion_summary_display)


,Channel,Total Leads,Total Opportunities,Won Opportunities,Lead → Opportunity Rate,Opportunity Win Rate,Lead → Won Rate,Largest Bottleneck
0,Paid Social,253,89,32,35.2%,36.0%,12.6%,Lead → Opportunity
1,Display,170,50,29,29.4%,58.0%,17.1%,Lead → Opportunity
2,Paid Search,44,16,7,36.4%,43.8%,15.9%,Lead → Opportunity


### Key Findings

- **Paid Social** generated the highest acquisition volume, producing **253 leads**, **89 opportunities**, and **32 won opportunities**, followed by **Display** (**170 → 50 → 29**) and **Paid Search** (**44 → 16 → 7**). Despite these differences in scale, all three channels experienced their largest drop-off during the Lead → Opportunity stage.

- At the channel level, **Display** achieved the strongest **Opportunity Win Rate (58.0%)**, substantially outperforming **Paid Search (43.8%)** and **Paid Social (36.0%)**. However, Display also recorded the lowest **Lead → Opportunity Rate (29.4%)**, indicating that its primary opportunity lies in improving the progression of leads into qualified opportunities while maintaining strong downstream sales conversion.

    - The funnel analysis shows that Display's superior marketing efficiency was driven primarily by a substantially higher Opportunity → Customer conversion rate (58%), despite having the lowest Lead → Opportunity conversion rate. This suggests that Display generated fewer opportunities relative to leads, but those opportunities converted to customers more effectively. The available data does not explain why Display outperformed the other channels, so this finding should be interpreted as a characteristic of this dataset rather than a broader conclusion.
        - Display achieved both the highest ROAS and the lowest Cost per Won Opportunity because it generated strong downstream business outcomes relative to its marketing investment.

- Campaign-level performance varied considerably. **Lead → Opportunity Rates** ranged from **17% to 48%**, while **Opportunity Win Rates** ranged from **17% to 78%**, highlighting substantial differences in funnel performance across individual campaigns, even within the same acquisition channel.

- Overall, the analysis indicates that the **Lead → Opportunity stage represents the largest conversion bottleneck across all acquisition channels**, whereas opportunities progressed to customers at comparatively higher rates. This suggests that increasing the proportion of leads progressing into qualified opportunities offers the greatest potential to improve overall funnel performance, while further investigation of high- and low-performing campaigns may help identify the drivers of these conversion differences.

## Step 6: Lead Cohort Quality
- Are newer lead cohorts converting into customers and revenue more effectively?
- **Important:** recent cohorts have had less time to progress through the sales cycle. Hence, their customer conversion and won revenue should therefore be interpreted cautiously.
    - Lower customer conversion or won revenue in newer cohorts may reflect cohort immaturity rather than weaker acquisition quality.

In [6]:
# ------------------------------------------------------------
# 6.1 Load monthly lead cohort data
#
# Ratios and weighted averages are recalculated from the
# underlying aggregated values rather than averaged directly.
# ------------------------------------------------------------

cohort_performance = conn.execute(
    """
    SELECT
        CAST(
            DATE_TRUNC('month', lead_created_month)
            AS DATE
        ) AS lead_created_month,

        SUM(total_leads) AS total_leads,
        SUM(total_opportunities) AS total_opportunities,
        SUM(total_customers) AS total_customers,

        SUM(pipeline_value) AS pipeline_value,
        SUM(won_revenue) AS won_revenue,

        CASE
            WHEN SUM(total_customers) > 0
                THEN SUM(won_revenue)
                     / SUM(total_customers)
        END AS average_deal_size,

        CASE
            WHEN SUM(total_customers) > 0
                THEN SUM(
                    average_sales_cycle_days
                    * total_customers
                )
                / SUM(total_customers)
        END AS average_sales_cycle_days,

        CASE
            WHEN SUM(total_leads) > 0
                THEN SUM(total_opportunities) * 1.0
                     / SUM(total_leads)
        END AS lead_to_opportunity_rate,

        CASE
            WHEN SUM(total_opportunities) > 0
                THEN SUM(total_customers) * 1.0
                     / SUM(total_opportunities)
        END AS opportunity_to_customer_rate

    FROM mart_lead_cohort_progression

    GROUP BY 1
    ORDER BY 1
    """
).df()


# ------------------------------------------------------------
# Data preparation
# ------------------------------------------------------------

cohort_performance["lead_created_month"] = pd.to_datetime(
    cohort_performance["lead_created_month"]
)


numeric_columns = [
    "total_leads",
    "total_opportunities",
    "total_customers",
    "pipeline_value",
    "won_revenue",
    "average_deal_size",
    "average_sales_cycle_days",
    "lead_to_opportunity_rate",
    "opportunity_to_customer_rate",
]


cohort_performance[numeric_columns] = (
    cohort_performance[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
)


cohort_performance = (
    cohort_performance[
        cohort_performance["total_leads"] > 0
    ]
    .sort_values("lead_created_month")
    .reset_index(drop=True)
)


cohort_performance["cohort_month"] = (
    cohort_performance["lead_created_month"]
    .dt.strftime("%b %Y")
)


# ------------------------------------------------------------
# Shared chart styling
# ------------------------------------------------------------

def style_cohort_chart(
    fig,
    title,
    height=550,
):
    layout = deepcopy(CHART_STYLE)

    layout["title"]["text"] = f"<b>{title}</b>"
    layout["height"] = height

    fig.update_layout(**layout)

    fig.update_xaxes(
        title_text=None,
        tickformat="%b %Y",
        tickangle=-45,
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=6,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    fig.update_yaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    return fig


# ============================================================
# VISUAL 1: COHORT CONVERSION PERFORMANCE
#
# Compares the conversion quality of each monthly lead cohort:
# - Lead → Opportunity Rate
# - Opportunity → Customer Rate
# ============================================================

fig_cohort_conversion = go.Figure()


fig_cohort_conversion.add_trace(
    go.Scatter(
        x=cohort_performance["lead_created_month"],
        y=cohort_performance[
            "lead_to_opportunity_rate"
        ],
        mode="lines+markers+text",
        name="Lead → Opportunity Rate",
        text=cohort_performance[
            "lead_to_opportunity_rate"
        ],
        line=dict(
            color="#FF6692",  # Pink / magenta
            dash="dot",
            width=2,
        ),
        texttemplate="%{text:.0%}",
        textposition="top center",
        cliponaxis=False,
        customdata=cohort_performance[
            [
                "total_leads",
                "total_opportunities",
                "total_customers",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y} Cohort</b><br>"
            "Lead → Opportunity Rate: %{y:.1%}<br>"
            "Total Leads: %{customdata[0]:,.0f}<br>"
            "Total Opportunities: %{customdata[1]:,.0f}<br>"
            "Total Customers: %{customdata[2]:,.0f}"
            "<extra></extra>"
        ),
    )
)


fig_cohort_conversion.add_trace(
    go.Scatter(
        x=cohort_performance["lead_created_month"],
        y=cohort_performance[
            "opportunity_to_customer_rate"
        ],
        mode="lines+markers+text",
        name="Opportunity → Customer Rate",
        text=cohort_performance[
            "opportunity_to_customer_rate"
        ],
        line=dict(
            color="#19D3F3",  # Cyan
            dash="dot",
            width=2,
        ),
        texttemplate="%{text:.0%}",
        textposition="bottom center",
        cliponaxis=False,
        customdata=cohort_performance[
            [
                "total_leads",
                "total_opportunities",
                "total_customers",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y} Cohort</b><br>"
            "Opportunity → Customer Rate: %{y:.1%}<br>"
            "Total Leads: %{customdata[0]:,.0f}<br>"
            "Total Opportunities: %{customdata[1]:,.0f}<br>"
            "Total Customers: %{customdata[2]:,.0f}"
            "<extra></extra>"
        ),
    )
)


conversion_axis_max = cohort_performance[
    [
        "lead_to_opportunity_rate",
        "opportunity_to_customer_rate",
    ]
].max().max()


style_cohort_chart(
    fig_cohort_conversion,
    "Lead Cohort Conversion Performance",
    height=570,
)


fig_cohort_conversion.update_layout(
    hovermode="x unified",
    legend=dict(
        title=dict(
            text="<b>Metrics</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(
        l=75,
        r=45,
        t=80,
        b=80,
    ),
)


fig_cohort_conversion.update_yaxes(
    title_text="<b>Conversion Rate</b>",
    tickformat=".0%",
    range=[
        0,
        conversion_axis_max,
    ],
)


fig_cohort_conversion.show()


# ============================================================
# VISUAL 2: COHORT COMMERCIAL OUTCOMES
#
# Bars:
# - Pipeline Value
# - Won Revenue
#
# Line:
# - Average Deal Size
#
# This compares the commercial value generated by each cohort.
# ============================================================

fig_cohort_commercial = make_subplots(
    specs=[
        [
            {
                "secondary_y": True,
            }
        ]
    ]
)


fig_cohort_commercial.add_trace(
    go.Bar(
        x=cohort_performance["lead_created_month"],
        y=cohort_performance["pipeline_value"],
        name="Pipeline Value",
        marker_color="#AB63FA",  # Purple
        customdata=cohort_performance[
            [
                "won_revenue",
                "total_opportunities",
                "total_customers",
                "average_sales_cycle_days",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y} Cohort</b><br>"
            "Pipeline Value: €%{y:,.0f}<br>"
            "Won Revenue: €%{customdata[0]:,.0f}<br>"
            "Total Opportunities: %{customdata[1]:,.0f}<br>"
            "Total Customers: %{customdata[2]:,.0f}<br>"
            "Average Sales Cycle: %{customdata[3]:.0f} days"
            "<extra></extra>"
        ),
    ),
    secondary_y=False,
)


fig_cohort_commercial.add_trace(
    go.Bar(
        x=cohort_performance["lead_created_month"],
        y=cohort_performance["won_revenue"],
        name="Won Revenue",
        marker_color="#00CC96",  # Green
        customdata=cohort_performance[
            [
                "pipeline_value",
                "total_customers",
                "average_deal_size",
                "average_sales_cycle_days",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y} Cohort</b><br>"
            "Won Revenue: €%{y:,.0f}<br>"
            "Pipeline Value: €%{customdata[0]:,.0f}<br>"
            "Total Customers: %{customdata[1]:,.0f}<br>"
            "Average Deal Size: €%{customdata[2]:,.0f}<br>"
            "Average Sales Cycle: %{customdata[3]:.0f} days"
            "<extra></extra>"
        ),
    ),
    secondary_y=False,
)


fig_cohort_commercial.add_trace(
    go.Scatter(
        x=cohort_performance["lead_created_month"],
        y=cohort_performance["average_deal_size"],
        mode="lines+markers+text",
        name="Average Deal Size",
        line=dict(
            color="#FF6692",  # Pink / magenta
            dash="dot",
            width=2,
        ),
        text=cohort_performance["average_deal_size"],
        texttemplate="€%{text:.3s}",
        textposition="top center",
        cliponaxis=False,
        customdata=cohort_performance[
            [
                "won_revenue",
                "total_customers",
                "average_sales_cycle_days",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y} Cohort</b><br>"
            "Average Deal Size: €%{y:,.0f}<br>"
            "Won Revenue: €%{customdata[0]:,.0f}<br>"
            "Total Customers: %{customdata[1]:,.0f}<br>"
            "Average Sales Cycle: %{customdata[2]:.0f} days"
            "<extra></extra>"
        ),
    ),
    secondary_y=True,
)


style_cohort_chart(
    fig_cohort_commercial,
    "Lead Cohort Revenue Performance",
    height=590,
)


fig_cohort_commercial.update_layout(
    barmode="group",
    bargap=0.20,
    bargroupgap=0.05,
    hovermode="x unified",
    legend=dict(
        title=dict(
            text="<b>Metrics</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(
        l=80,
        r=90,
        t=80,
        b=80,
    ),
)


fig_cohort_commercial.update_yaxes(
    title_text="<b>Pipeline / Revenue</b>",
    tickprefix="€",
    tickformat="~s",
    rangemode="tozero",
    secondary_y=False,
)


fig_cohort_commercial.update_yaxes(
    title_text="<b>Average Deal Size</b>",
    tickprefix="€",
    tickformat="~s",
    rangemode="tozero",
    secondary_y=True,
)


fig_cohort_commercial.show()


# ============================================================
# 6.2 COHORT PERFORMANCE SUMMARY TABLE
#
# Includes the supporting volume, conversion, commercial value
# and sales-cycle metrics for each lead cohort.
# ============================================================

cohort_summary = cohort_performance[
    [
        "lead_created_month",
        "total_leads",
        "total_opportunities",
        "total_customers",
        "pipeline_value",
        "won_revenue",
        "average_deal_size",
        "average_sales_cycle_days",
        "lead_to_opportunity_rate",
        "opportunity_to_customer_rate",
    ]
].copy()


cohort_summary["lead_created_month"] = (
    cohort_summary["lead_created_month"]
    .dt.strftime("%b %Y")
)


cohort_summary.columns = [
    "Lead Cohort",
    "Total Leads",
    "Total Opportunities",
    "Total Customers",
    "Pipeline Value",
    "Won Revenue",
    "Average Deal Size",
    "Average Sales Cycle",
    "Lead → Opportunity Rate",
    "Opportunity → Customer Rate",
]


cohort_summary_display = (
    cohort_summary
    .style
    .format(
        {
            "Total Leads": "{:,.0f}",
            "Total Opportunities": "{:,.0f}",
            "Total Customers": "{:,.0f}",
            "Pipeline Value": "€{:,.0f}",
            "Won Revenue": "€{:,.0f}",
            "Average Deal Size": "€{:,.0f}",
            "Average Sales Cycle": "{:.0f} days",
            "Lead → Opportunity Rate": "{:.1%}",
            "Opportunity → Customer Rate": "{:.1%}",
        },
        na_rep="—",
    )
    .set_properties(
        **{
            "font-family": "Calibri",
            "text-align": "center",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("font-family", "Calibri"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("background-color", "#F5F5F5"),
                    ("white-space", "nowrap"),
                ],
            },
            {
                "selector": "td:first-child",
                "props": [
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("white-space", "nowrap"),
                ],
            },
        ]
    )
)


display(cohort_summary_display)


,Lead Cohort,Total Leads,Total Opportunities,Total Customers,Pipeline Value,Won Revenue,Average Deal Size,Average Sales Cycle,Lead → Opportunity Rate,Opportunity → Customer Rate
0,Jan 2025,51,18,8,"€372,187","€190,236","€23,780",59 days,35.3%,44.4%
1,Feb 2025,59,22,11,"€458,241","€241,756","€21,978",51 days,37.3%,50.0%
2,Mar 2025,66,21,11,"€415,656","€222,992","€20,272",54 days,31.8%,52.4%
3,Apr 2025,48,15,3,"€296,472","€43,560","€14,520",72 days,31.2%,20.0%
4,May 2025,56,14,3,"€320,725","€72,186","€24,062",66 days,25.0%,21.4%
5,Jun 2025,57,22,11,"€368,556","€201,829","€18,348",50 days,38.6%,50.0%
6,Jul 2025,57,16,5,"€303,605","€97,577","€19,515",49 days,28.1%,31.2%
7,Aug 2025,51,18,11,"€375,544","€249,227","€22,657",62 days,35.3%,61.1%
8,Sep 2025,55,16,8,"€295,953","€139,259","€17,407",32 days,29.1%,50.0%


### Key Findings

- Lead cohort performance varied across the reporting period, with no consistent upward or downward trend in downstream conversion or revenue generation. **Lead → Opportunity Rates** ranged from **25.0% to 38.6%**, while **Opportunity → Customer Rates** varied more substantially, from **20.0% to 61.1%**, indicating meaningful differences in cohort performance.

- Pipeline generation remained relatively stable across cohorts, generally ranging between **€296K and €458K**. However, the proportion of pipeline converted into won revenue varied considerably. For example, the **August 2025 cohort** generated the highest **won revenue (€249K)** and the highest **Opportunity → Customer Rate (61.1%)**, while the **April** and **May** cohorts recorded the weakest downstream conversion (**20.0%** and **21.4%**, respectively), resulting in substantially lower won revenue despite generating pipeline values comparable to other cohorts.

    - April and May were the weakest-performing months. Marketing investment remained relatively stable, while Lead → Opportunity and Opportunity → Customer conversion rates declined, sales cycles lengthened, and won revenue fell significantly. The largest deterioration occurred in converting opportunities into customers, indicating that the primary bottleneck was in the later stages of the sales funnel. The available data does not contain sufficient business context to determine the underlying cause. Further investigation would require additional operational or business data.

- Average deal size remained relatively stable across most cohorts (approximately **€17K–24K**), while downstream conversion rates varied considerably. This suggests that differences in revenue performance were associated with variation in cohort conversion as well as the number of customers generated, rather than substantial differences in average deal value.

- Recent cohorts should be interpreted cautiously, as they have had less time to progress through the sales cycle. Consequently, observed customer conversion rates and won revenue for newer cohorts may continue to change as opportunities mature, and should not be interpreted as definitive measures of acquisition quality.

## Step 7: Customer Product Engagement
- How is monthly product activity changing over time, and how does it differ across plan tiers?
- Analyze monthly trends in active users and active companies, then compare product activity across plan tiers to understand how engagement varies across customer segments.
- Analysis period: January 2025 through September 2025.
    - Months after September are excluded to align the analysis with the complete observation period. Later months contain partial product-usage data and could create misleading trends

In [7]:
# ------------------------------------------------------------
# 7.1 Load product engagement data
# ------------------------------------------------------------

product_adoption = conn.execute(
    """
    SELECT
        event_month,
        aggregation_level,
        plan_tier,
        active_users,
        active_companies

    FROM mart_product_adoption

    ORDER BY
        event_month,
        aggregation_level,
        plan_tier
    """
).df()


# ------------------------------------------------------------
# Data preparation
# ------------------------------------------------------------

product_adoption["event_month"] = pd.to_datetime(
    product_adoption["event_month"]
)


numeric_columns = [
    "active_users",
    "active_companies",
]


product_adoption[numeric_columns] = (
    product_adoption[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
)


product_adoption["aggregation_level"] = (
    product_adoption["aggregation_level"]
    .fillna("unknown")
    .str.lower()
)


product_adoption["plan_tier"] = (
    product_adoption["plan_tier"]
    .fillna("unknown")
    .str.replace("_", " ", regex=False)
    .str.title()
)


product_adoption = (
    product_adoption
    .sort_values(
        [
            "event_month",
            "aggregation_level",
            "plan_tier",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Analysis-period filter
# ------------------------------------------------------------

analysis_end_date = pd.Timestamp("2025-09-30")


product_adoption = (
    product_adoption[
        product_adoption["event_month"] <= analysis_end_date
    ]
    .copy()
    .reset_index(drop=True)
)


# Separate overall monthly totals from plan-tier rows.
#
# Overall totals must not be calculated by summing plan-tier
# rows because a user or company may appear in more than one
# plan tier during the same month.

monthly_overall = (
    product_adoption[
        product_adoption["aggregation_level"] == "overall"
    ]
    .copy()
    .sort_values("event_month")
    .reset_index(drop=True)
)


monthly_by_plan = (
    product_adoption[
        product_adoption["aggregation_level"] == "plan_tier"
    ]
    .copy()
    .sort_values(
        [
            "event_month",
            "plan_tier",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Shared chart styling
# ------------------------------------------------------------

def style_product_adoption_chart(
    fig,
    title,
    height=550,
):
    layout = deepcopy(CHART_STYLE)

    layout["title"]["text"] = f"<b>{title}</b>"
    layout["height"] = height

    fig.update_layout(**layout)

    fig.update_xaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=6,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    fig.update_yaxes(
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor="#BDBDBD",
        ticks="outside",
        ticklen=5,
        tickwidth=1,
        tickcolor="#BDBDBD",
        zeroline=False,
    )

    return fig


# ============================================================
# VISUAL 1: OVERALL MONTHLY PRODUCT ACTIVITY
#
# Shows the number of distinct users and companies that
# generated at least one tracked product event each month.
# ============================================================

fig_monthly_activity = go.Figure()


fig_monthly_activity.add_trace(
    go.Scatter(
        x=monthly_overall["event_month"],
        y=monthly_overall["active_users"],
        mode="lines+markers+text",
        name="Monthly Active Users",
        text=monthly_overall["active_users"],
        texttemplate="%{text:,.0f}",
        textposition="top center",
        line=dict(
            dash="dot",
            width=2,
        ),
        marker=dict(
            size=7,
        ),
        customdata=monthly_overall[
            [
                "active_companies",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y}</b><br>"
            "Monthly Active Users: %{y:,.0f}<br>"
            "Monthly Active Companies: %{customdata[0]:,.0f}"
            "<extra></extra>"
        ),
    )
)


fig_monthly_activity.add_trace(
    go.Scatter(
        x=monthly_overall["event_month"],
        y=monthly_overall["active_companies"],
        mode="lines+markers+text",
        name="Monthly Active Companies",
        text=monthly_overall["active_companies"],
        texttemplate="%{text:,.0f}",
        textposition="bottom center",
        line=dict(
            dash="dot",
            width=2,
        ),
        marker=dict(
            size=7,
        ),
        customdata=monthly_overall[
            [
                "active_users",
            ]
        ],
        hovertemplate=(
            "<b>%{x|%b %Y}</b><br>"
            "Monthly Active Companies: %{y:,.0f}<br>"
            "Monthly Active Users: %{customdata[0]:,.0f}"
            "<extra></extra>"
        ),
    )
)


style_product_adoption_chart(
    fig_monthly_activity,
    "Overall Monthly Product Activity",
    height=570,
)


fig_monthly_activity.update_layout(
    hovermode="x unified",
    legend=dict(
        title=dict(
            text="<b>Metrics</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(
        l=75,
        r=45,
        t=80,
        b=75,
    ),
)


fig_monthly_activity.update_xaxes(
    title_text=None,
    tickformat="%b %Y",
    tickangle=-45,
)


fig_monthly_activity.update_yaxes(
    title_text="<b>Distinct Active Entities</b>",
    tickformat="~s",
    rangemode="tozero",
)


fig_monthly_activity.show()


# ============================================================
# VISUAL 2: MONTHLY ACTIVE USERS BY PLAN TIER
#
# Compares distinct active-user trends across plan tiers over
# the complete analysis period.
# ============================================================

fig_users_by_plan = go.Figure()


for plan_tier in sorted(
    monthly_by_plan["plan_tier"].dropna().unique()
):

    plan_data = (
        monthly_by_plan[
            monthly_by_plan["plan_tier"] == plan_tier
        ]
        .copy()
        .sort_values("event_month")
    )

    fig_users_by_plan.add_trace(
        go.Scatter(
            x=plan_data["event_month"],
            y=plan_data["active_users"],
            mode="lines+markers",
            name=plan_tier,
            line=dict(
                dash="dot",
                width=2,
            ),
            marker=dict(
                size=7,
            ),
            customdata=plan_data[
                [
                    "active_companies",
                ]
            ],
            hovertemplate=(
                f"<b>{plan_tier}</b><br>"
                "%{x|%b %Y}<br>"
                "Monthly Active Users: %{y:,.0f}<br>"
                "Monthly Active Companies: %{customdata[0]:,.0f}"
                "<extra></extra>"
            ),
        )
    )


style_product_adoption_chart(
    fig_users_by_plan,
    "Monthly Active Users by Plan Tier",
    height=570,
)


fig_users_by_plan.update_layout(
    hovermode="x unified",
    legend=dict(
        title=dict(
            text="<b>Plan Tier</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(
        l=75,
        r=45,
        t=80,
        b=75,
    ),
)


fig_users_by_plan.update_xaxes(
    title_text=None,
    tickformat="%b %Y",
    tickangle=-45,
)


fig_users_by_plan.update_yaxes(
    title_text="<b>Monthly Active Users</b>",
    tickformat="~s",
    rangemode="tozero",
)


fig_users_by_plan.show()


# ============================================================
# VISUAL 3: MONTHLY ACTIVE COMPANIES BY PLAN TIER
#
# Compares distinct active-company trends across plan tiers
# over the complete analysis period.
# ============================================================

fig_companies_by_plan = go.Figure()


for plan_tier in sorted(
    monthly_by_plan["plan_tier"].dropna().unique()
):

    plan_data = (
        monthly_by_plan[
            monthly_by_plan["plan_tier"] == plan_tier
        ]
        .copy()
        .sort_values("event_month")
    )

    fig_companies_by_plan.add_trace(
        go.Scatter(
            x=plan_data["event_month"],
            y=plan_data["active_companies"],
            mode="lines+markers",
            name=plan_tier,
            line=dict(
                dash="dot",
                width=2,
            ),
            marker=dict(
                size=7,
            ),
            customdata=plan_data[
                [
                    "active_users",
                ]
            ],
            hovertemplate=(
                f"<b>{plan_tier}</b><br>"
                "%{x|%b %Y}<br>"
                "Monthly Active Companies: %{y:,.0f}<br>"
                "Monthly Active Users: %{customdata[0]:,.0f}"
                "<extra></extra>"
            ),
        )
    )


style_product_adoption_chart(
    fig_companies_by_plan,
    "Monthly Active Companies by Plan Tier",
    height=570,
)


fig_companies_by_plan.update_layout(
    hovermode="x unified",
    legend=dict(
        title=dict(
            text="<b>Plan Tier</b>",
        ),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(
        l=75,
        r=45,
        t=80,
        b=75,
    ),
)


fig_companies_by_plan.update_xaxes(
    title_text=None,
    tickformat="%b %Y",
    tickangle=-45,
)


fig_companies_by_plan.update_yaxes(
    title_text="<b>Monthly Active Companies</b>",
    tickformat="~s",
    rangemode="tozero",
)


fig_companies_by_plan.show()

### Key Findings

- Overall product activity increased over the reporting period. **Monthly Active Users** grew from **422 in January** to **1,067 in September**, while **Monthly Active Companies** increased from **30 to 62**. Although activity fluctuated slightly after March, both metrics remained consistently above their early-year levels, indicating sustained product engagement throughout the reporting period.

- Product engagement differed across plan tiers. **Starter** consistently recorded the highest monthly active users, while **Starter** and **Pro** maintained the largest number of active companies throughout most of the reporting period. **Enterprise** exhibited relatively stable engagement, whereas **Trial** activity fluctuated more noticeably across both users and companies.

- Trends in monthly active users and monthly active companies were generally aligned across plan tiers. Periods of higher user activity were typically accompanied by higher numbers of active companies, suggesting that increases in product engagement occurred across both individual users and customer organizations.

## Step 8: Summary of Findings and Recommendations

### Summary of Findings

- The business generated a **substantial and relatively stable sales pipeline**, but attributed won revenue remained below advertising spend, resulting in an overall **Paid ROAS of 0.89x**. While marketing investment remained relatively stable throughout the reporting period, fluctuations in won revenue were more closely aligned with changes in downstream funnel conversion than with advertising spend.

- **Display** delivered the strongest observed acquisition efficiency during the reporting period, achieving the highest **Paid ROAS (6.62x)** together with the lowest **Cost per Won Opportunity (€3.0K)** and consistently dominating the highest-performing campaign rankings. In contrast, **Paid Search** and several **Paid Social** campaigns generated substantially weaker returns and considerably higher acquisition costs despite greater marketing investment.

- The Lead → Opportunity Rate represented the largest and most consistent bottleneck across all acquisition channels. In comparison, Opportunity Win Rate was generally stronger, indicating that improving the progression of leads into qualified opportunities offers the greatest potential to increase won revenue while improving overall marketing efficiency.

- Lead cohorts generated relatively similar pipeline values but materially different customer conversion and revenue outcomes. Average won deal values remained relatively stable across cohorts, suggesting that differences in business performance were primarily associated with funnel conversion rather than deal size.

- Product activity increased steadily throughout the reporting period, with growth observed in both monthly active users and monthly active companies across plan tiers. This indicates continued product adoption following customer acquisition, although the available data does not support evaluating long-term customer retention.

### Recommendations

1. **Validate whether Display performance is maintained as investment increases.**  
   Display consistently delivered the strongest acquisition efficiency during the reporting period. Before reallocating a larger share of marketing budget, perform an incremental budget scaling test on the highest-performing Display campaigns while monitoring Paid ROAS, Cost per Won Opportunity, and Opportunity Win Rate to determine whether campaign efficiency can be sustained at higher spend levels.

2. **Prioritize improving Lead → Opportunity conversion.**  
   The Lead → Opportunity Rate was the largest and most consistent bottleneck across every acquisition channel. Use campaign- and cohort-level analysis to identify where leads fail to progress into qualified opportunities, and use these insights to prioritize improvements across the acquisition funnel.

3. **Evaluate campaign performance using downstream business outcomes.**  
   Campaigns within the same acquisition channel exhibited substantial variation in Paid ROAS, Cost per Won Opportunity, and conversion performance. Regularly comparing high- and low-performing campaigns using downstream business outcomes—not just lead volume—may help identify which campaigns consistently generate the strongest revenue outcomes.

4. **Monitor lead cohorts until they fully mature.**  
   Lead cohorts produced similar pipeline values but materially different customer conversion and won revenue outcomes. Continue tracking cohort performance over time to distinguish temporary timing effects from sustained changes in acquisition quality and overall business performance.

## Step 9: Limitations and Future Enhancements

### Limitations

The provided datasets support analysis of **marketing performance, lead progression, and product engagement**, but they do not capture the complete customer journey or the information required for more advanced business and growth analytics.

| Limitation | Business Impact | Additional Data Required |
|------------|-----------------|--------------------------|
| **Marketing Attribution** | Marketing performance is attributed using the lead creation date because no ad-click or attribution timestamp is available. Recent campaigns naturally show lower Won Revenue, Paid ROAS, and conversion rates until opportunities mature, making the model more suitable for historical performance reporting than short-term marketing optimization. | Ad-click timestamp, attribution timestamp or click identifier, and multi-touch attribution history. |
| **Website Analytics** | The customer journey cannot be analysed prior to lead creation. Website behaviour such as landing page engagement, session flow, and form abandonment cannot be evaluated, limiting analysis of top-of-funnel conversion and marketing effectiveness before lead generation. | Website analytics events (e.g., sessions, page views, landing pages, conversions), visitor identifiers, and lead-to-visitor mapping. |
| **Product Activation** | Trial participation is used as a proxy for customer activation because the dataset does not define a business activation event. As a result, product adoption may not fully reflect meaningful customer activation. | Defined product activation event, activation criteria, and onboarding or feature adoption milestones. |
| **Customer Lifecycle** | Subscription lifecycle data is unavailable, preventing analysis of customer retention, expansion, and long-term customer value. Metrics such as MRR, ARR, NRR, GRR, Expansion Revenue, and Customer Lifetime Value (LTV) cannot be calculated. | Subscription history, recurring revenue, renewals, cancellations, expansion and contraction revenue, and churn events. |
| **Customer Segmentation** | Performance cannot be compared across customer groups, limiting insights into acquisition, conversion, and product adoption for different customer segments. | Company attributes such as industry, company size, geography, and customer segment. |
| **Seat Utilization** | Product events capture seat-related events but not licensed seat capacity, preventing reliable measurement of workspace utilization. | Licensed seat capacity and active seat allocation over time. |

---

### Future Enhancements

With additional business data, the analytics layer could be extended to support:

- Multi-touch marketing attribution.
    - The current model attributes each lead to a single campaign because only one campaign identifier is available in the CRM. With richer customer journey data, the model could be extended to support multi-touch attribution, allowing conversion credit to be distributed across multiple marketing interactions rather than a single campaign.
- Product activation metrics based on defined activation milestones.
    - The dataset includes product usage events but lacks an explicit activation definition, activation milestones, or an is_activated indicator, preventing measurement of product activation and its impact on retention and revenue. 
- Customer lifecycle analytics, including MRR, ARR, NRR, GRR, Expansion Revenue, and Customer Lifetime Value (LTV).
- Customer segmentation by industry, company size, geography, and customer segment.
- Seat utilization and workspace adoption metrics.

## Pipeline Check

In [8]:
pipeline_end = datetime.now()

print("\n" + "=" * 80)
print("Pipeline completed successfully.")
print(f"Started : {pipeline_start:%Y-%m-%d %H:%M:%S}")
print(f"Finished: {pipeline_end:%Y-%m-%d %H:%M:%S}")
print(f"Duration: {pipeline_end - pipeline_start}")
print("=" * 80)

conn.close()

print("DuckDB connection closed.")


Pipeline completed successfully.
Started : 2026-08-10 13:41:08
Finished: 2026-08-10 13:41:13
Duration: 0:00:05.235646
DuckDB connection closed.
